# Phase 3: learner profile discovery and validation

Run 3A, 3B, and 3C in order. Use the same source CSV throughout. Phase 2 results enable additional SEM-weighted sensitivity configurations.


In [ ]:
import os
import zipfile
import tempfile
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

INPUT_DATA_PATH = os.environ.get('PHASE3_INPUT_DATA', '/content/drive/MyDrive/SOICT-2026/data/soict_2026_survey_responses.csv')
PHASE2_RESULTS_PATH = os.environ.get('PHASE3_PHASE2_RESULTS', '')
OUTPUT_BASE_DIR = os.environ.get('PHASE3_OUTPUT_DIR', '/content/drive/MyDrive/SOICT-2026/results/phase_03')
FAST_TEST = os.environ.get('PHASE3_FAST_TEST', '0') == '1'
SELECTED_K = 2  # Phase 3B evaluates the prespecified two-profile solution.

input_source = Path(INPUT_DATA_PATH).expanduser()
if not input_source.is_file():
    raise FileNotFoundError(f'Phase 3 input data not found: {input_source}')
output_root = Path(OUTPUT_BASE_DIR).expanduser()
output_root.mkdir(parents=True, exist_ok=True)
if input_source.suffix.lower() != '.csv':
    raise ValueError('INPUT_DATA_PATH must point to a CSV with the 71-item study schema.')
INPUT_CSV_PATH = str(input_source)
print('Input CSV:', input_source)
print('Output directory:', output_root)
print('Fast smoke test:', FAST_TEST)


In [ ]:
# Load primary-model standardized coefficients and validate phase 2 provenance.
import io
import hashlib

def _phase2_member(path, filename):
    if path.is_dir():
        matches = [p for p in path.rglob(filename) if 'Monte_Carlo' not in p.parts]
        if len(matches) != 1:
            raise ValueError(f'Expected exactly one match for {filename} in the phase 2 results.')
        return matches[0].read_bytes()
    with zipfile.ZipFile(path) as archive:
        matches = [name for name in archive.namelist()
                   if Path(name).name == filename and 'Monte_Carlo' not in Path(name).parts]
        if len(matches) != 1:
            raise ValueError(f'Expected exactly one match for {filename} in the phase 2 ZIP.')
        return archive.read(matches[0])

def read_phase3_source(path):
    for enc in ('utf-8-sig', 'utf-8', 'cp1258', 'cp1252', 'latin1'):
        try:
            with open(path, encoding=enc) as handle:
                first_line = handle.readline()
            for sep in sorted((',', ';', '\t'), key=first_line.count, reverse=True):
                frame = pd.read_csv(path, encoding=enc, sep=sep)
                if len(frame.columns) > 1:
                    frame.columns = [str(x).replace('\ufeff', '').strip() for x in frame.columns]
                    return frame, enc, sep
        except (UnicodeError, ValueError, pd.errors.ParserError):
            continue
    raise ValueError(f'Unable to read input CSV: {path}')

SEM_BETA = None
SEM_PROVENANCE = {'type': 'unavailable', 'note': 'No matching phase 2 results; omit configurations M2 and M5.'}
if PHASE2_RESULTS_PATH:
    phase2_path = Path(PHASE2_RESULTS_PATH).expanduser()
    if not phase2_path.exists():
        raise FileNotFoundError(f'Phase 2 results not found: {phase2_path}')
    source, _, _ = read_phase3_source(INPUT_CSV_PATH)
    phase2_bytes = _phase2_member(phase2_path, '00_du_lieu_71_cau_cho_R.csv')
    phase2_data = pd.read_csv(io.BytesIO(phase2_bytes))
    item_columns = list(phase2_data.columns)
    if len(item_columns) != 71 or not set(item_columns).issubset(source.columns):
        raise ValueError('Phase 2 results do not contain the same 71 item codes as phase 3 input.')
    left = source[item_columns].apply(pd.to_numeric, errors='raise').reset_index(drop=True)
    right = phase2_data[item_columns].apply(pd.to_numeric, errors='raise').reset_index(drop=True)
    if left.shape != right.shape or not np.array_equal(left.to_numpy(), right.to_numpy(), equal_nan=True):
        raise ValueError('Phase 2 results do not match phase 3 responses and row order.')
    beta_bytes = _phase2_member(phase2_path, '04_he_so_duong_dan.csv')
    beta_table = pd.read_csv(io.BytesIO(beta_bytes))
    sub = beta_table.loc[(beta_table['mo_hinh'] == 'SEM_PARTIAL_PRIMARY') &
                         (beta_table['lhs'] == 'STRATEGY') & (beta_table['op'] == '~') &
                         beta_table['rhs'].isin(['ENP', 'PTED', 'MEU'])]
    if len(sub) != 3 or sub['rhs'].nunique() != 3:
        raise ValueError('Phase 2 results lack standardized tier I to STRATEGY paths.')
    SEM_BETA = dict(zip(sub['rhs'], sub['std.all'].astype(float)))
    if not all(np.isfinite(value) and value >= 0 for value in SEM_BETA.values()):
        raise ValueError('SEM weights are nonfinite or negative; select a different weighting scheme.')
    SEM_PROVENANCE = {'type': 'SEM_PARTIAL_PRIMARY', 'path': str(phase2_path),
                      'beta_sha256': hashlib.sha256(beta_bytes).hexdigest(),
                      'data_sha256': hashlib.sha256(phase2_bytes).hexdigest(),
                      'standardized_beta': SEM_BETA}
print('Tier I weight source:', SEM_PROVENANCE['type'])


## 3A. Select candidate cluster counts


In [ ]:
OUTPUT_BASE_DIR = str(output_root / '3A')

In [ ]:
K_VALUES = list(range(2, 5)) if FAST_TEST else list(range(2, 9))
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Python dependencies are ready.")
RANDOM_SEED = 20260911
KMEANS_N_INIT = 20 if FAST_TEST else 100
PAM_N_INIT = 5 if FAST_TEST else 25
GAP_REFERENCE_SAMPLES = 8 if FAST_TEST else 50
STABILITY_RESAMPLES = 30 if FAST_TEST else 500
STABILITY_SUBSAMPLE_FRACTION = 0.80
MIN_CLUSTER_PROPORTION = 0.05


In [ ]:
import hashlib
import json
import math
import platform
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.metrics.cluster import contingency_matrix
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "text.color": "black",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

SCALE_ITEMS = {
    "ENP": [f"ENP{i:02d}" for i in range(1, 7)],
    "PTED": [f"PTED{i:02d}" for i in range(1, 7)],
    "MEU": [f"MEU{i:02d}" for i in range(1, 6)],
    "REH": [f"REH{i:02d}" for i in range(1, 4)],
    "ELA": [f"ELA{i:02d}" for i in range(1, 4)],
    "ORG": [f"ORG{i:02d}" for i in range(1, 4)],
    "CRT": [f"CRT{i:02d}" for i in range(1, 4)],
    "MSR": [f"MSR{i:02d}" for i in range(1, 5)],
    "TSEM": [f"TSEM{i:02d}" for i in range(1, 4)],
    "VLS": [f"VLS{i:02d}" for i in range(1, 7)],
    "DEL": [f"DEL{i:02d}" for i in range(1, 8)],
    "LPSN": [f"LPSN{i:02d}" for i in range(1, 6)],
    "SFN": [f"SFN{i:02d}" for i in range(1, 5)],
    "TECN": [f"TECN{i:02d}" for i in range(1, 7)],
    "PU": [f"PU{i:02d}" for i in range(1, 5)],
    "BI": [f"BI{i:02d}" for i in range(1, 4)],
}
ALL_ITEMS = [item for items in SCALE_ITEMS.values() for item in items]
TIER1_SCALES = ["ENP", "PTED", "MEU"]
TIER1_NO_ENP = ["PTED", "MEU"]
TIER2_SCALES = ["REH", "ELA", "ORG", "CRT", "MSR", "TSEM", "VLS", "DEL"]
PRIMARY_FEATURES = TIER1_SCALES + TIER2_SCALES
SENSITIVITY_FEATURES = TIER1_NO_ENP + TIER2_SCALES
HELD_OUT_SCALES = ["LPSN", "SFN", "TECN", "PU", "BI"]


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_csv_robust(path):
    encodings = ["utf-8-sig", "utf-8", "cp1258", "cp1252", "latin1"]
    last_error = None
    for encoding in encodings:
        try:
            with open(path, "r", encoding=encoding) as handle:
                first_line = handle.readline()
            separators = sorted([",", ";", "\t"], key=lambda s: first_line.count(s), reverse=True)
            for separator in separators:
                try:
                    frame = pd.read_csv(path, encoding=encoding, sep=separator)
                    if frame.shape[1] > 1:
                        return frame, encoding, separator
                except Exception as error:
                    last_error = error
        except Exception as error:
            last_error = error
    raise RuntimeError(f"Cannot read CSV with supported encodings and delimiters: {last_error}")


def save_csv(frame, path):
    frame.to_csv(path, index=False, encoding="utf-8-sig")


def within_cluster_ss(X, labels):
    total = 0.0
    for label in np.unique(labels):
        block = X[labels == label]
        center = block.mean(axis=0)
        total += float(np.square(block - center).sum())
    return max(total, np.finfo(float).tiny)


def cluster_metrics(X, labels):
    counts = np.bincount(labels)
    return {
        "silhouette": float(silhouette_score(X, labels)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels)),
        "davies_bouldin": float(davies_bouldin_score(X, labels)),
        "wcss": within_cluster_ss(X, labels),
        "min_cluster_n": int(counts.min()),
        "max_cluster_n": int(counts.max()),
        "min_cluster_pct": float(counts.min() / len(labels)),
    }


def kmeans_labels(X, k, seed, n_init):
    model = KMeans(n_clusters=k, random_state=seed, n_init=n_init, algorithm="lloyd")
    return model.fit_predict(X), model


def initial_medoids(distance_matrix, k, rng):
    n = distance_matrix.shape[0]
    selected = [int(rng.integers(n))]
    while len(selected) < k:
        nearest = distance_matrix[:, selected].min(axis=1)
        nearest[selected] = 0.0
        weights = nearest ** 2
        if weights.sum() <= 0:
            candidates = np.setdiff1d(np.arange(n), np.array(selected))
            selected.append(int(rng.choice(candidates)))
        else:
            selected.append(int(rng.choice(np.arange(n), p=weights / weights.sum())))
    return np.array(selected, dtype=int)


def pam_labels(X, k, seed, n_init=20, max_iter=100):
    D = pairwise_distances(X, metric="euclidean")
    master_rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_init):
        rng = np.random.default_rng(int(master_rng.integers(0, 2**31 - 1)))
        medoids = initial_medoids(D, k, rng)
        for _ in range(max_iter):
            labels = np.argmin(D[:, medoids], axis=1)
            new_medoids = medoids.copy()
            for cluster_id in range(k):
                members = np.flatnonzero(labels == cluster_id)
                if len(members) == 0:
                    candidates = np.setdiff1d(np.arange(len(X)), new_medoids)
                    new_medoids[cluster_id] = int(rng.choice(candidates))
                else:
                    sub = D[np.ix_(members, members)]
                    new_medoids[cluster_id] = int(members[np.argmin(sub.sum(axis=1))])
            if np.array_equal(new_medoids, medoids):
                break
            medoids = new_medoids
        labels = np.argmin(D[:, medoids], axis=1)
        objective = float(D[np.arange(len(X)), medoids[labels]].sum())
        if best is None or objective < best[0]:
            best = (objective, labels.copy(), medoids.copy())
    return best[1], best[2], best[0]


def align_labels(reference, candidate):
    matrix = contingency_matrix(reference, candidate)
    row_ind, col_ind = linear_sum_assignment(-matrix)
    mapping = {candidate_label: reference_label for reference_label, candidate_label in zip(row_ind, col_ind)}
    return np.array([mapping.get(label, label) for label in candidate], dtype=int)


def cluster_jaccard(reference, candidate):
    aligned = align_labels(reference, candidate)
    scores = []
    for label in np.unique(reference):
        a = reference == label
        b = aligned == label
        union = np.logical_or(a, b).sum()
        scores.append(float(np.logical_and(a, b).sum() / union) if union else np.nan)
    return scores

In [ ]:
input_path = Path(INPUT_CSV_PATH)
output_base = Path(OUTPUT_BASE_DIR)
if not input_path.is_file():
    raise FileNotFoundError(f"Input file not found: {input_path}")
output_base.mkdir(parents=True, exist_ok=True)

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir = output_base / f"notebook_3A_chan_doan_phan_cum_{run_stamp}"
run_dir.mkdir(parents=False, exist_ok=False)

input_hash_before = sha256_file(input_path)
df, detected_encoding, detected_separator = read_csv_robust(input_path)
df.columns = [str(column).replace("\ufeff", "").strip() for column in df.columns]

if df.columns.duplicated().any():
    raise AssertionError("Input column names must be unique.")
missing_columns = [column for column in ALL_ITEMS if column not in df.columns]
if missing_columns:
    raise AssertionError(f"Required columns missing: {missing_columns}")
if 'RESPONDENT_ID' not in df.columns:
    df.insert(0, 'RESPONDENT_ID', [f'ROW_{row:08d}' for row in range(1, len(df) + 1)])
if df["RESPONDENT_ID"].isna().any() or df["RESPONDENT_ID"].duplicated().any():
    raise AssertionError("RESPONDENT_ID must be present and unique.")

invalid_rows = []
for column in ALL_ITEMS:
    converted = pd.to_numeric(df[column], errors="coerce")
    nonempty_original = df[column].notna() & df[column].astype(str).str.strip().ne("")
    failed_conversion = nonempty_original & converted.isna()
    if failed_conversion.any():
        invalid_rows.append({"column": column, "problem": "cannot be parsed as a number", "count": int(failed_conversion.sum())})
    invalid_value = converted.notna() & ~converted.isin([1, 2, 3, 4, 5])
    if invalid_value.any():
        invalid_rows.append({"column": column, "problem": "outside the 1 to 5 range", "count": int(invalid_value.sum())})
    df[column] = converted
if invalid_rows:
    raise AssertionError(f"Invalid values: {invalid_rows}")

enp_items = SCALE_ITEMS["ENP"]
enp_all_missing = df[enp_items].isna().all(axis=1)
enp_any_missing = df[enp_items].isna().any(axis=1)
enp_partial_missing = enp_any_missing & ~enp_all_missing
if enp_partial_missing.any():
    raise AssertionError("An ENP response block is only partly observed.")

non_enp_items = [item for item in ALL_ITEMS if item not in enp_items]
if int(df[non_enp_items].isna().sum().sum()) != 0:
    raise AssertionError("An item outside ENP is missing.")
if int(df[SCALE_ITEMS["PU"] + SCALE_ITEMS["BI"]].isna().sum().sum()) != 0:
    raise AssertionError("PU or BI has missing responses.")

input_hash_after = sha256_file(input_path)
if input_hash_before != input_hash_after:
    raise AssertionError("Input changed while the notebook was reading it.")

audit = {
    "input_path": str(input_path),
    "sha256": input_hash_before,
    "encoding": detected_encoding,
    "separator": {",": "comma", ";": "semicolon", "\t": "tab"}.get(detected_separator, detected_separator),
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "likert_items": int(len(ALL_ITEMS)),
    "complete_71": int((~df[ALL_ITEMS].isna().any(axis=1)).sum()),
    "full_enp_missing": int(enp_all_missing.sum()),
    "missing_outside_enp": int(df[non_enp_items].isna().sum().sum()),
    "pu_bi_missing": int(df[SCALE_ITEMS["PU"] + SCALE_ITEMS["BI"]].isna().sum().sum()),
}

with open(run_dir / "00_kiem_tra_dau_vao.json", "w", encoding="utf-8") as handle:
    json.dump(audit, handle, ensure_ascii=False, indent=2)
print(json.dumps(audit, ensure_ascii=False, indent=2))
print("Results directory:", run_dir)
EXPECTED_ROWS = len(df)
EXPECTED_COMPLETE_ENP = int((~df[ALL_ITEMS].isna().any(axis=1)).sum())
EXPECTED_ENP_MISSING = int(enp_all_missing.sum())
if EXPECTED_COMPLETE_ENP < max(K_VALUES) + 1 or EXPECTED_ROWS < max(K_VALUES) + 1:
    raise ValueError("Too few complete ENP profiles for the configured K range.")


In [ ]:
score_df = pd.DataFrame({"RESPONDENT_ID": df["RESPONDENT_ID"].astype(str)})
for scale, items in SCALE_ITEMS.items():
    score_df[scale] = df[items].mean(axis=1, skipna=False)

for optional in ["GENDER", "FIELD_OF_STUDY", "S0_ENGLISH_NECESSARY"]:
    if optional in df.columns:
        score_df[optional] = df[optional]

primary_mask = score_df[PRIMARY_FEATURES].notna().all(axis=1)
primary_scores = score_df.loc[primary_mask, ["RESPONDENT_ID", *PRIMARY_FEATURES]].reset_index(drop=True)
sensitivity_scores = score_df[["RESPONDENT_ID", *SENSITIVITY_FEATURES]].copy().reset_index(drop=True)
if len(primary_scores) != EXPECTED_COMPLETE_ENP:
    raise AssertionError(f"Primary sample should contain {EXPECTED_COMPLETE_ENP} respondents; found {len(primary_scores)}")
if len(sensitivity_scores) != EXPECTED_ROWS:
    raise AssertionError("The sensitivity sample has too few rows.")

descriptive = score_df[list(SCALE_ITEMS)].describe().T.reset_index().rename(columns={"index": "scale"})
descriptive["missing_n"] = [int(score_df[scale].isna().sum()) for scale in descriptive["scale"]]
correlation = score_df[list(SCALE_ITEMS)].corr(method="spearman")

standardizer_primary = StandardScaler()
X_primary_equal = standardizer_primary.fit_transform(primary_scores[PRIMARY_FEATURES])
X_primary_balanced = X_primary_equal.copy()
X_primary_balanced[:, :len(TIER1_SCALES)] /= math.sqrt(len(TIER1_SCALES))
X_primary_balanced[:, len(TIER1_SCALES):] /= math.sqrt(len(TIER2_SCALES))

standardizer_sensitivity = StandardScaler()
X_sensitivity_equal = standardizer_sensitivity.fit_transform(sensitivity_scores[SENSITIVITY_FEATURES])
X_sensitivity_balanced = X_sensitivity_equal.copy()
X_sensitivity_balanced[:, :len(TIER1_NO_ENP)] /= math.sqrt(len(TIER1_NO_ENP))
X_sensitivity_balanced[:, len(TIER1_NO_ENP):] /= math.sqrt(len(TIER2_SCALES))

outlier_z = np.max(np.abs(X_primary_equal), axis=1)
outlier_flags = pd.DataFrame({
    "RESPONDENT_ID": primary_scores["RESPONDENT_ID"],
    "max_abs_z": outlier_z,
    "flag_abs_z_over_3_5": outlier_z > 3.5,
})

primary_export = primary_scores.copy()
for index, feature in enumerate(PRIMARY_FEATURES):
    primary_export[f"Z_EQUAL_{feature}"] = X_primary_equal[:, index]
    primary_export[f"Z_BALANCED_{feature}"] = X_primary_balanced[:, index]
sensitivity_export = sensitivity_scores.copy()
for index, feature in enumerate(SENSITIVITY_FEATURES):
    sensitivity_export[f"Z_EQUAL_{feature}"] = X_sensitivity_equal[:, index]
    sensitivity_export[f"Z_BALANCED_{feature}"] = X_sensitivity_balanced[:, index]

save_csv(score_df, run_dir / "01_diem_16_thang.csv")
save_csv(descriptive, run_dir / "02_thong_ke_mo_ta_diem_thang.csv")
correlation.to_csv(run_dir / "03_tuong_quan_spearman_16_thang.csv", encoding="utf-8-sig")
save_csv(primary_export, run_dir / "04_dau_vao_phan_cum_chinh_MAU_CHINH.csv")
save_csv(sensitivity_export, run_dir / "05_dau_vao_do_nhay_MAU_DAY_DU_khong_ENP.csv")
save_csv(outlier_flags, run_dir / "06_co_ngoai_le_khong_tu_dong_loai.csv")

print("Primary sample:", primary_scores.shape)
print("Sensitivity sample:", sensitivity_scores.shape)
print("Respondents flagged as outliers:", int(outlier_flags["flag_abs_z_over_3_5"].sum()))
print(descriptive[["scale", "count", "mean", "std", "min", "max", "missing_n"]].to_string(index=False))

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
diagnostic_rows = []
kmeans_primary_labels = {}
kmeans_primary_models = {}
pam_primary_labels = {}
ward_primary_labels = {}
kmeans_equal_labels = {}
kmeans_sensitivity_labels = {}

for k in K_VALUES:
    labels, model = kmeans_labels(X_primary_balanced, k, RANDOM_SEED + k, KMEANS_N_INIT)
    kmeans_primary_labels[k] = labels
    kmeans_primary_models[k] = model
    diagnostic_rows.append({"sample": "MAU_CHINH_ENP_BALANCED", "algorithm": "KMEANS", "k": k, **cluster_metrics(X_primary_balanced, labels)})

    labels_pam, medoids, objective = pam_labels(X_primary_balanced, k, RANDOM_SEED + 1000 + k, PAM_N_INIT)
    pam_primary_labels[k] = labels_pam
    diagnostic_rows.append({"sample": "MAU_CHINH_ENP_BALANCED", "algorithm": "PAM", "k": k, **cluster_metrics(X_primary_balanced, labels_pam), "pam_total_distance": objective})

    labels_ward = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X_primary_balanced)
    ward_primary_labels[k] = labels_ward
    diagnostic_rows.append({"sample": "MAU_CHINH_ENP_BALANCED", "algorithm": "WARD", "k": k, **cluster_metrics(X_primary_balanced, labels_ward)})

    equal_labels, _ = kmeans_labels(X_primary_equal, k, RANDOM_SEED + 2000 + k, KMEANS_N_INIT)
    kmeans_equal_labels[k] = equal_labels
    diagnostic_rows.append({"sample": "MAU_CHINH_ENP_EQUAL", "algorithm": "KMEANS", "k": k, **cluster_metrics(X_primary_equal, equal_labels)})

    sensitivity_labels, _ = kmeans_labels(X_sensitivity_balanced, k, RANDOM_SEED + 3000 + k, KMEANS_N_INIT)
    kmeans_sensitivity_labels[k] = sensitivity_labels
    diagnostic_rows.append({"sample": "MAU_DAY_DU_NO_ENP_BALANCED", "algorithm": "KMEANS", "k": k, **cluster_metrics(X_sensitivity_balanced, sensitivity_labels)})

diagnostics = pd.DataFrame(diagnostic_rows)

# Gap statistic for primary K-Means analysis.
gap_rows = []
mins = X_primary_balanced.min(axis=0)
maxs = X_primary_balanced.max(axis=0)
for k in K_VALUES:
    observed_wcss = within_cluster_ss(X_primary_balanced, kmeans_primary_labels[k])
    reference_logs = []
    for b in range(GAP_REFERENCE_SAMPLES):
        reference = rng.uniform(mins, maxs, size=X_primary_balanced.shape)
        ref_labels, _ = kmeans_labels(reference, k, RANDOM_SEED + 10000 + 100 * k + b, min(20, KMEANS_N_INIT))
        reference_logs.append(math.log(within_cluster_ss(reference, ref_labels)))
    reference_logs = np.array(reference_logs)
    gap_rows.append({
        "k": k,
        "gap": float(reference_logs.mean() - math.log(observed_wcss)),
        "gap_se": float(reference_logs.std(ddof=1) * math.sqrt(1 + 1 / GAP_REFERENCE_SAMPLES)),
    })
gap_df = pd.DataFrame(gap_rows)
gap_rule = []
for index in range(len(gap_df) - 1):
    current = gap_df.iloc[index]
    following = gap_df.iloc[index + 1]
    if current["gap"] >= following["gap"] - following["gap_se"]:
        gap_rule.append(int(current["k"]))
gap_df["gap_rule_candidate"] = gap_df["k"].isin(gap_rule)

# Agreement across algorithms, weighting, and sample definitions.
sensitivity_id_to_index = {value: index for index, value in enumerate(sensitivity_scores["RESPONDENT_ID"])}
primary_in_sensitivity = np.array([sensitivity_id_to_index[value] for value in primary_scores["RESPONDENT_ID"]])
agreement_rows = []
for k in K_VALUES:
    agreement_rows.append({
        "k": k,
        "ARI_KMEANS_PAM": adjusted_rand_score(kmeans_primary_labels[k], pam_primary_labels[k]),
        "ARI_KMEANS_WARD": adjusted_rand_score(kmeans_primary_labels[k], ward_primary_labels[k]),
        "ARI_BALANCED_EQUAL_WEIGHT": adjusted_rand_score(kmeans_primary_labels[k], kmeans_equal_labels[k]),
        "ARI_MAU_CHINH_WITH_ENP_vs_MAU_DAY_DU_NO_ENP": adjusted_rand_score(
            kmeans_primary_labels[k], kmeans_sensitivity_labels[k][primary_in_sensitivity]
        ),
    })
agreement_df = pd.DataFrame(agreement_rows)

labels_export = pd.DataFrame({"RESPONDENT_ID": primary_scores["RESPONDENT_ID"]})
for k in K_VALUES:
    labels_export[f"KMEANS_K{k}"] = kmeans_primary_labels[k] + 1
    labels_export[f"PAM_K{k}"] = pam_primary_labels[k] + 1
    labels_export[f"WARD_K{k}"] = ward_primary_labels[k] + 1

centroid_rows = []
for k in K_VALUES:
    labels = kmeans_primary_labels[k]
    for cluster in range(k):
        members = labels == cluster
        for feature_index, feature in enumerate(PRIMARY_FEATURES):
            centroid_rows.append({
                "k": k,
                "cluster": cluster + 1,
                "feature": feature,
                "mean_z_equal": float(X_primary_equal[members, feature_index].mean()),
                "mean_raw": float(primary_scores.loc[members, feature].mean()),
                "n": int(members.sum()),
            })
centroids = pd.DataFrame(centroid_rows)

save_csv(diagnostics, run_dir / "07_chi_so_chan_doan_cac_thuat_toan.csv")
save_csv(gap_df, run_dir / "08_gap_statistic_kmeans_MAU_CHINH.csv")
save_csv(agreement_df, run_dir / "09_do_dong_thuan_thuat_toan_trong_so_co_mau.csv")
save_csv(labels_export, run_dir / "10_nhan_cum_chan_doan_chua_chot.csv")
save_csv(centroids, run_dir / "11_trung_tam_cum_chan_doan.csv")

print("Cluster count diagnostics completed.")
print(diagnostics.query("sample == 'MAU_CHINH_ENP_BALANCED' and algorithm == 'KMEANS'").to_string(index=False))
print("Gap-rule candidates:", gap_rule if gap_rule else "No eligible K before the maximum tested")

In [ ]:
stability_rows = []
stability_rng = np.random.default_rng(RANDOM_SEED + 50000)
n = len(X_primary_balanced)
subsample_n = int(round(n * STABILITY_SUBSAMPLE_FRACTION))

for k in K_VALUES:
    reference_labels = kmeans_primary_labels[k]
    ari_values = []
    jaccard_values = []
    for iteration in range(STABILITY_RESAMPLES):
        subset = np.sort(stability_rng.choice(n, size=subsample_n, replace=False))
        model = KMeans(
            n_clusters=k,
            random_state=RANDOM_SEED + 60000 + k * 1000 + iteration,
            n_init=min(20, KMEANS_N_INIT),
            algorithm="lloyd",
        )
        model.fit(X_primary_balanced[subset])
        candidate_labels = model.predict(X_primary_balanced)
        ari = adjusted_rand_score(reference_labels, candidate_labels)
        cluster_scores = cluster_jaccard(reference_labels, candidate_labels)
        ari_values.append(ari)
        jaccard_values.append(cluster_scores)
        row = {"k": k, "iteration": iteration + 1, "ARI": ari}
        for cluster_index, value in enumerate(cluster_scores, 1):
            row[f"JACCARD_CLUSTER_{cluster_index}"] = value
        stability_rows.append(row)

stability_detail = pd.DataFrame(stability_rows)
stability_summary_rows = []
for k in K_VALUES:
    block = stability_detail[stability_detail["k"] == k]
    cluster_columns = [column for column in block.columns if column.startswith("JACCARD_CLUSTER_") and block[column].notna().any()]
    cluster_means = block[cluster_columns].mean()
    stability_summary_rows.append({
        "k": k,
        "ARI_mean": float(block["ARI"].mean()),
        "ARI_median": float(block["ARI"].median()),
        "ARI_p025": float(block["ARI"].quantile(0.025)),
        "ARI_p975": float(block["ARI"].quantile(0.975)),
        "Jaccard_mean_all_clusters": float(cluster_means.mean()),
        "Jaccard_min_cluster_mean": float(cluster_means.min()),
    })
stability_summary = pd.DataFrame(stability_summary_rows)

primary_kmeans = diagnostics.query("sample == 'MAU_CHINH_ENP_BALANCED' and algorithm == 'KMEANS'").copy()
decision_table = (
    primary_kmeans.merge(gap_df, on="k", how="left")
    .merge(stability_summary, on="k", how="left")
    .merge(agreement_df, on="k", how="left")
)
decision_table["small_cluster_flag"] = decision_table["min_cluster_pct"] < MIN_CLUSTER_PROPORTION
decision_table["rank_silhouette"] = decision_table["silhouette"].rank(ascending=False, method="min")
decision_table["rank_calinski"] = decision_table["calinski_harabasz"].rank(ascending=False, method="min")
decision_table["rank_davies"] = decision_table["davies_bouldin"].rank(ascending=True, method="min")
decision_table["rank_stability"] = decision_table["ARI_mean"].rank(ascending=False, method="min")
decision_table["mean_reference_rank"] = decision_table[
    ["rank_silhouette", "rank_calinski", "rank_davies", "rank_stability"]
].mean(axis=1)
decision_table["automatic_final_decision"] = "KHONG_TU_DONG_CHOT"

save_csv(stability_detail, run_dir / "12_on_dinh_lay_mau_lai_chi_tiet.csv")
save_csv(stability_summary, run_dir / "13_on_dinh_lay_mau_lai_tom_tat.csv")
save_csv(decision_table, run_dir / "14_bang_bang_chung_lua_chon_K.csv")

print("Completed", STABILITY_RESAMPLES, "resamples for each K.")
print(decision_table[[
    "k", "silhouette", "calinski_harabasz", "davies_bouldin", "gap", "gap_rule_candidate",
    "ARI_mean", "Jaccard_min_cluster_mean", "min_cluster_n", "small_cluster_flag",
    "ARI_KMEANS_PAM", "ARI_BALANCED_EQUAL_WEIGHT", "ARI_MAU_CHINH_WITH_ENP_vs_MAU_DAY_DU_NO_ENP",
    "mean_reference_rank", "automatic_final_decision",
]].to_string(index=False))

In [ ]:
primary_diag = decision_table.sort_values("k")

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
plots = [
    ("wcss", "Elbow: within-cluster sum of squares", "Lower; inspect the elbow"),
    ("silhouette", "Silhouette", "Higher is better"),
    ("calinski_harabasz", "Calinski–Harabasz", "Higher is better"),
    ("davies_bouldin", "Davies–Bouldin", "Lower is better"),
]
for axis, (column, title, ylabel) in zip(axes.ravel(), plots):
    axis.plot(primary_diag["k"], primary_diag[column], color="black", marker="o")
    axis.set_title(title, fontweight="bold")
    axis.set_xlabel("Number of clusters K")
    axis.set_ylabel(ylabel)
    axis.grid(color="0.85", linewidth=0.8)
fig.tight_layout()
fig.savefig(run_dir / "15_bieu_do_chan_doan_K.png", dpi=200, bbox_inches="tight")
plt.close(fig)

fig, axis = plt.subplots(figsize=(7.5, 4.8))
axis.errorbar(gap_df["k"], gap_df["gap"], yerr=gap_df["gap_se"], color="black", marker="o", capsize=4)
for k in gap_rule:
    row = gap_df[gap_df["k"] == k].iloc[0]
    axis.scatter([k], [row["gap"]], marker="s", s=70, facecolors="white", edgecolors="black", zorder=3)
axis.set_title("K-Means gap statistic", fontweight="bold")
axis.set_xlabel("Number of clusters K")
axis.set_ylabel("Gap and standard error")
axis.grid(color="0.85")
fig.tight_layout()
fig.savefig(run_dir / "16_bieu_do_gap_statistic.png", dpi=200, bbox_inches="tight")
plt.close(fig)

fig, axis = plt.subplots(figsize=(7.5, 4.8))
axis.plot(stability_summary["k"], stability_summary["ARI_mean"], color="black", marker="o", label="Mean ARI")
axis.fill_between(stability_summary["k"], stability_summary["ARI_p025"], stability_summary["ARI_p975"], color="0.85", label="2.5% to 97.5% interval")
axis.plot(stability_summary["k"], stability_summary["Jaccard_min_cluster_mean"], color="black", linestyle="--", marker="s", label="Minimum cluster Jaccard")
axis.set_ylim(0, 1.02)
axis.set_title("Resampling stability", fontweight="bold")
axis.set_xlabel("Number of clusters K")
axis.set_ylabel("Stability")
axis.legend(frameon=False)
axis.grid(color="0.85")
fig.tight_layout()
fig.savefig(run_dir / "17_bieu_do_do_on_dinh.png", dpi=200, bbox_inches="tight")
plt.close(fig)

fig, axis = plt.subplots(figsize=(8.5, 5.2))
for column, marker, linestyle in [
    ("ARI_KMEANS_PAM", "o", "-"),
    ("ARI_KMEANS_WARD", "s", "--"),
    ("ARI_BALANCED_EQUAL_WEIGHT", "^", "-."),
    ("ARI_MAU_CHINH_WITH_ENP_vs_MAU_DAY_DU_NO_ENP", "d", ":"),
]:
    axis.plot(agreement_df["k"], agreement_df[column], color="black", marker=marker, linestyle=linestyle, label=column)
axis.set_ylim(-0.05, 1.02)
axis.set_title("Agreement across clustering algorithms", fontweight="bold")
axis.set_xlabel("Number of clusters K")
axis.set_ylabel("Adjusted Rand Index")
axis.legend(frameon=False, fontsize=8)
axis.grid(color="0.85")
fig.tight_layout()
fig.savefig(run_dir / "18_bieu_do_do_nhay_va_dong_thuan.png", dpi=200, bbox_inches="tight")
plt.close(fig)

Z = linkage(X_primary_balanced, method="ward")
fig, axis = plt.subplots(figsize=(11, 5.5))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=8, color_threshold=0, above_threshold_color="black", ax=axis)
axis.set_title("Reduced Ward dendrogram", fontweight="bold")
axis.set_xlabel("Reduced respondent set")
axis.set_ylabel("Merge distance")
fig.tight_layout()
fig.savefig(run_dir / "19_cay_phan_cum_Ward.png", dpi=200, bbox_inches="tight")
plt.close(fig)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
projection = pca.fit_transform(X_primary_balanced)
fig, axes = plt.subplots(3, 3, figsize=(12, 11))
for axis, k in zip(axes.ravel(), K_VALUES):
    labels = kmeans_primary_labels[k]
    axis.scatter(projection[:, 0], projection[:, 1], c=labels, cmap="Greys", s=13, edgecolors="none")
    axis.set_title(f"K = {k}", fontweight="bold")
    axis.set_xlabel("PCA 1")
    axis.set_ylabel("PCA 2")
for axis in axes.ravel()[len(K_VALUES):]:
    axis.axis("off")
fig.suptitle("PCA projection of K-Means candidates", fontweight="bold")
fig.tight_layout()
fig.savefig(run_dir / "20_PCA_cac_nghiem_K.png", dpi=200, bbox_inches="tight")
plt.close(fig)

fig, axes = plt.subplots(4, 2, figsize=(13, 15))
for axis, k in zip(axes.ravel(), K_VALUES):
    block = centroids[centroids["k"] == k].pivot(index="cluster", columns="feature", values="mean_z_equal")[PRIMARY_FEATURES]
    image = axis.imshow(block.values, cmap="Greys", aspect="auto", vmin=-1.5, vmax=1.5)
    axis.set_title(f"K = {k}", fontweight="bold")
    axis.set_xticks(range(len(PRIMARY_FEATURES)), PRIMARY_FEATURES, rotation=45, ha="right")
    axis.set_yticks(range(len(block.index)), [f"Cluster {value}" for value in block.index])
color_axis = axes.ravel()[-1]
fig.colorbar(image, cax=color_axis, label="Mean z-score")
color_axis.set_title("Color scale", fontweight="bold")
fig.suptitle("Diagnostic cluster centers; profile names are not assigned here", fontweight="bold")
fig.subplots_adjust(top=0.95, right=0.88, hspace=0.55, wspace=0.30)
fig.savefig(run_dir / "21_ban_do_trung_tam_cum.png", dpi=200)
plt.close(fig)

print("Seven diagnostic figures were saved.")

In [ ]:
best_silhouette = int(primary_diag.loc[primary_diag["silhouette"].idxmax(), "k"])
best_calinski = int(primary_diag.loc[primary_diag["calinski_harabasz"].idxmax(), "k"])
best_davies = int(primary_diag.loc[primary_diag["davies_bouldin"].idxmin(), "k"])
best_stability = int(stability_summary.loc[stability_summary["ARI_mean"].idxmax(), "k"])
ranked_reference = decision_table.sort_values(["mean_reference_rank", "k"])["k"].astype(int).tolist()

summary_text = f'''# Phase 3A: diagnostic results

## Scope and data

- Primary sample: {len(primary_scores)} respondents with complete ENP responses.
- Sensitivity sample: {len(sensitivity_scores)} respondents; ENP is omitted.
- Primary clustering scales: {', '.join(PRIMARY_FEATURES)}.
- Tiers III and IV are excluded from cluster formation.
- Input: {len(df)} rows and {df.shape[1]} columns; {len(ALL_ITEMS)} Likert items.
- Complete 71-item respondents: {audit['complete_71']}.
- Respondents routed away from ENP: {audit['full_enp_missing']}.
- Missing cells outside ENP: {audit['missing_outside_enp']}; missing PU and BI cells: {audit['pu_bi_missing']}.
- Respondents flagged with an absolute z-score above 3.5: {int(outlier_flags['flag_abs_z_over_3_5'].sum())}. No automatic exclusions were made.

## Cluster-count diagnostics

- Highest silhouette at K = {best_silhouette}.
- Highest Calinski-Harabasz at K = {best_calinski}.
- Lowest Davies-Bouldin at K = {best_davies}.
- Highest mean resampled ARI at K = {best_stability}.
- Gap-rule candidates: {gap_rule if gap_rule else 'no eligible K before the maximum tested'}.
- Reference ordering by mean rank across four metrics: {ranked_reference}.

The metrics may favor different cluster counts. Examine resampling ARI and Jaccard, a minimum cluster proportion of {MIN_CLUSTER_PROPORTION:.0%}, agreement with PAM and Ward, the effect of weighting and excluding ENP, and whether centers differ in shape rather than only level. Weak evidence does not justify forcing a partition.

## Files to inspect

Input audit, cluster diagnostics, gap statistic, algorithm agreement, resampling stability, K assessment, and diagnostic figures 15 through 21 are saved with the run manifest.
'''

with open(run_dir / "22_huong_dan_doc_ket_qua.md", "w", encoding="utf-8") as handle:
    handle.write(summary_text)

run_config = {
    "created_utc": run_stamp,
    "python_version": platform.python_version(),
    "random_seed": RANDOM_SEED,
    "k_values": K_VALUES,
    "kmeans_n_init": KMEANS_N_INIT,
    "pam_n_init": PAM_N_INIT,
    "gap_reference_samples": GAP_REFERENCE_SAMPLES,
    "stability_resamples": STABILITY_RESAMPLES,
    "stability_subsample_fraction": STABILITY_SUBSAMPLE_FRACTION,
    "min_cluster_proportion": MIN_CLUSTER_PROPORTION,
    "fast_test": FAST_TEST,
    "automatic_final_decision": False,
}
with open(run_dir / "23_cau_hinh_lan_chay.json", "w", encoding="utf-8") as handle:
    json.dump(run_config, handle, ensure_ascii=False, indent=2)

# Verify core result files by reading them back.
readback_checks = []
for filename, expected_rows in [
    ("01_diem_16_thang.csv", EXPECTED_ROWS),
    ("04_dau_vao_phan_cum_chinh_MAU_CHINH.csv", EXPECTED_COMPLETE_ENP),
    ("05_dau_vao_do_nhay_MAU_DAY_DU_khong_ENP.csv", EXPECTED_ROWS),
    ("07_chi_so_chan_doan_cac_thuat_toan.csv", len(K_VALUES) * 5),
    ("14_bang_bang_chung_lua_chon_K.csv", len(K_VALUES)),
]:
    frame = pd.read_csv(run_dir / filename, encoding="utf-8-sig")
    ok = len(frame) == expected_rows
    readback_checks.append({"file": filename, "expected_rows": expected_rows, "actual_rows": len(frame), "passed": ok})
    if not ok:
        raise AssertionError(f"Readback failed: {filename}")
readback = pd.DataFrame(readback_checks)
save_csv(readback, run_dir / "24_kiem_tra_doc_lai_dau_ra.csv")

# The manifest does not include a hash of itself.
manifest_rows = []
for path in sorted(run_dir.iterdir()):
    if path.is_file() and path.name != "25_manifest_sha256.csv":
        manifest_rows.append({"file": path.name, "size_bytes": path.stat().st_size, "sha256": sha256_file(path)})
manifest = pd.DataFrame(manifest_rows)
save_csv(manifest, run_dir / "25_manifest_sha256.csv")

if sha256_file(input_path) != input_hash_before:
    raise AssertionError("Input changed before the notebook finished.")

print(summary_text)
print("Phase 3A complete. No cluster count was selected automatically.")
print("Results directory:", run_dir)

In [ ]:
INPUT_NOTEBOOK3A_DIR = str(run_dir)
PHASE3_3A_DIR = str(run_dir)
print('3A results:', PHASE3_3A_DIR)

## 3B. Validate the prespecified two-profile solution


In [ ]:
OUTPUT_BASE_DIR = str(output_root / '3B')
CLUSTER_COLUMN = f'KMEANS_K{SELECTED_K}'
if SELECTED_K not in K_VALUES:
    raise ValueError('SELECTED_K must lie within the 3A scan range.')

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Python dependencies are ready.")
RANDOM_SEED = 20260911
BOOTSTRAP_RESAMPLES = 300 if FAST_TEST else 5000
PERMUTATION_RESAMPLES = 500 if FAST_TEST else 5000
SILHOUETTE_REFERENCE_TOLERANCE = 1e-9


In [ ]:
import hashlib
import json
import math
import platform
import shutil
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, ttest_ind
from sklearn.linear_model import LinearRegression
from sklearn.metrics import silhouette_samples

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "text.color": "black",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

TIER1_SCALES = ["ENP", "PTED", "MEU"]
TIER2_SCALES = ["REH", "ELA", "ORG", "CRT", "MSR", "TSEM", "VLS", "DEL"]
CLUSTER_FEATURES = TIER1_SCALES + TIER2_SCALES
EXTERNAL_SCALES = ["LPSN", "SFN", "TECN", "PU", "BI"]

REQUIRED_FILES = [
    "01_diem_16_thang.csv",
    "04_dau_vao_phan_cum_chinh_MAU_CHINH.csv",
    "10_nhan_cum_chan_doan_chua_chot.csv",
    "13_on_dinh_lay_mau_lai_tom_tat.csv",
    "14_bang_bang_chung_lua_chon_K.csv",
    "25_manifest_sha256.csv",
]


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def save_csv(frame, path):
    frame.to_csv(path, index=False, encoding="utf-8-sig")


def hedges_g(group_high, group_low):
    high = np.asarray(group_high, dtype=float)
    low = np.asarray(group_low, dtype=float)
    n_high, n_low = len(high), len(low)
    pooled_var = ((n_high - 1) * high.var(ddof=1) + (n_low - 1) * low.var(ddof=1)) / (n_high + n_low - 2)
    if pooled_var <= 0:
        return np.nan
    cohen_d = (high.mean() - low.mean()) / math.sqrt(pooled_var)
    correction = 1 - 3 / (4 * (n_high + n_low) - 9)
    return float(correction * cohen_d)


def effect_label(value):
    magnitude = abs(value)
    if magnitude < 0.20:
        return "RAT_NHO"
    if magnitude < 0.50:
        return "NHO"
    if magnitude < 0.80:
        return "TRUNG_BINH"
    return "LON"


def bootstrap_difference_and_g(high, low, repetitions, rng):
    high = np.asarray(high, dtype=float)
    low = np.asarray(low, dtype=float)
    differences = np.empty(repetitions)
    effects = np.empty(repetitions)
    for index in range(repetitions):
        high_sample = rng.choice(high, size=len(high), replace=True)
        low_sample = rng.choice(low, size=len(low), replace=True)
        differences[index] = high_sample.mean() - low_sample.mean()
        effects[index] = hedges_g(high_sample, low_sample)
    return {
        "difference_ci_low": float(np.nanquantile(differences, 0.025)),
        "difference_ci_high": float(np.nanquantile(differences, 0.975)),
        "hedges_g_ci_low": float(np.nanquantile(effects, 0.025)),
        "hedges_g_ci_high": float(np.nanquantile(effects, 0.975)),
    }


def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    ranked = p[order]
    adjusted = ranked * len(p) / np.arange(1, len(p) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    result = np.empty_like(adjusted)
    result[order] = adjusted
    return result


def permanova_two_groups(X, labels, repetitions, rng):
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)

    def statistic(current_labels):
        overall = X.mean(axis=0)
        total_ss = float(np.square(X - overall).sum())
        within_ss = 0.0
        unique = np.unique(current_labels)
        for label in unique:
            block = X[current_labels == label]
            within_ss += float(np.square(block - block.mean(axis=0)).sum())
        between_ss = total_ss - within_ss
        df_between = len(unique) - 1
        df_within = len(X) - len(unique)
        pseudo_f = (between_ss / df_between) / (within_ss / df_within)
        return pseudo_f, between_ss / total_ss

    observed_f, r_squared = statistic(labels)
    exceed = 0
    for _ in range(repetitions):
        permuted = rng.permutation(labels)
        permuted_f, _ = statistic(permuted)
        exceed += permuted_f >= observed_f
    p_value = (exceed + 1) / (repetitions + 1)
    return float(observed_f), float(r_squared), float(p_value)

In [ ]:
input_dir = Path(INPUT_NOTEBOOK3A_DIR)
output_base = Path(OUTPUT_BASE_DIR)
if not input_dir.is_dir():
    raise FileNotFoundError(f"Phase 3A result directory not found: {input_dir}")
for filename in REQUIRED_FILES:
    if not (input_dir / filename).is_file():
        raise FileNotFoundError(f"Required phase 3A result missing: {filename}")

output_base.mkdir(parents=True, exist_ok=True)
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir = output_base / f"notebook_3B_kiem_dinh_ho_so_{run_stamp}"
run_dir.mkdir(parents=False, exist_ok=False)

manifest_3a = pd.read_csv(input_dir / "25_manifest_sha256.csv", encoding="utf-8-sig")
manifest_map = dict(zip(manifest_3a["file"], manifest_3a["sha256"]))
hash_checks = []
for filename in REQUIRED_FILES[:-1]:
    actual_hash = sha256_file(input_dir / filename)
    expected_hash = manifest_map.get(filename)
    passed = expected_hash == actual_hash
    hash_checks.append({
        "file": filename,
        "expected_sha256": expected_hash,
        "actual_sha256": actual_hash,
        "passed": passed,
    })
    if not passed:
        raise AssertionError(f"File hash disagrees with the phase 3A manifest: {filename}")

scores = pd.read_csv(input_dir / "01_diem_16_thang.csv", encoding="utf-8-sig")
cluster_input = pd.read_csv(input_dir / "04_dau_vao_phan_cum_chinh_MAU_CHINH.csv", encoding="utf-8-sig")
labels = pd.read_csv(input_dir / "10_nhan_cum_chan_doan_chua_chot.csv", encoding="utf-8-sig")
stability = pd.read_csv(input_dir / "13_on_dinh_lay_mau_lai_tom_tat.csv", encoding="utf-8-sig")
decision_3a = pd.read_csv(input_dir / "14_bang_bang_chung_lua_chon_K.csv", encoding="utf-8-sig")

for name, frame in {"scores": scores, "cluster_input": cluster_input, "labels": labels}.items():
    if "RESPONDENT_ID" not in frame.columns:
        raise AssertionError(f"File {name} is missing RESPONDENT_ID.")
    if frame["RESPONDENT_ID"].isna().any() or frame["RESPONDENT_ID"].duplicated().any():
        raise AssertionError(f"RESPONDENT_ID is missing or not unique in {name}.")

EXPECTED_TOTAL_N = len(scores)
EXPECTED_PRIMARY_N = int(scores[CLUSTER_FEATURES].notna().all(axis=1).sum())
if len(scores) != EXPECTED_TOTAL_N:
    raise AssertionError(f"The scale score table must contain {EXPECTED_TOTAL_N} rows; found {len(scores)}")
if len(cluster_input) != EXPECTED_PRIMARY_N or len(labels) != EXPECTED_PRIMARY_N:
    raise AssertionError("Cluster data and assignments must match the ENP-complete sample size.")
if set(cluster_input["RESPONDENT_ID"]) != set(labels["RESPONDENT_ID"]):
    raise AssertionError("Respondent IDs differ between cluster data and assignments.")
if not set(cluster_input["RESPONDENT_ID"]).issubset(set(scores["RESPONDENT_ID"])):
    raise AssertionError("A clustering ID is absent from the scale score table.")
if CLUSTER_COLUMN not in labels.columns:
    raise AssertionError(f"Cluster label column is missing: {CLUSTER_COLUMN}.")
if set(labels[CLUSTER_COLUMN].dropna().unique()) != set(range(1, SELECTED_K + 1)):
    raise AssertionError("K-Means labels must contain every selected cluster.")

required_cluster_columns = [f"Z_EQUAL_{scale}" for scale in CLUSTER_FEATURES] + [f"Z_BALANCED_{scale}" for scale in CLUSTER_FEATURES]
missing_cluster_columns = [column for column in required_cluster_columns if column not in cluster_input.columns]
missing_external = [column for column in EXTERNAL_SCALES if column not in scores.columns]
if missing_cluster_columns or missing_external:
    raise AssertionError(f"Missing columns: {missing_cluster_columns + missing_external}")

analysis = (
    cluster_input
    .merge(labels[["RESPONDENT_ID", CLUSTER_COLUMN]], on="RESPONDENT_ID", how="left", validate="one_to_one")
    .merge(scores[["RESPONDENT_ID", *EXTERNAL_SCALES]], on="RESPONDENT_ID", how="left", validate="one_to_one")
)
if analysis[[CLUSTER_COLUMN, *EXTERNAL_SCALES]].isna().any().any():
    raise AssertionError("Cluster labels or external validation scales are missing.")

input_audit = {
    "input_notebook3a_dir": str(input_dir),
    "rows_scale_scores": int(len(scores)),
    "rows_primary_cluster_sample": int(len(cluster_input)),
    "rows_merged_analysis": int(len(analysis)),
    "selected_k": SELECTED_K,
    "cluster_column": CLUSTER_COLUMN,
    "unique_ids_scores": int(scores["RESPONDENT_ID"].nunique()),
    "unique_ids_analysis": int(analysis["RESPONDENT_ID"].nunique()),
    "hash_checks_passed": bool(all(row["passed"] for row in hash_checks)),
}
with open(run_dir / "00_kiem_tra_dau_vao.json", "w", encoding="utf-8") as handle:
    json.dump(input_audit, handle, ensure_ascii=False, indent=2)
save_csv(pd.DataFrame(hash_checks), run_dir / "00b_doi_chieu_ma_bam_3A.csv")
print(json.dumps(input_audit, ensure_ascii=False, indent=2))
print("Results directory:", run_dir)

In [ ]:
equal_t2_columns = [f"Z_EQUAL_{scale}" for scale in TIER2_SCALES]
analysis["T2_INTENSITY_Z"] = analysis[equal_t2_columns].mean(axis=1)
original_means = analysis.groupby(CLUSTER_COLUMN)["T2_INTENSITY_Z"].mean()
high_original_label = int(original_means.idxmax())
low_original_label = int(original_means.idxmin())

analysis = analysis.rename(columns={CLUSTER_COLUMN: "KMEANS_K2_ORIGINAL"})
analysis["PROFILE_ID"] = np.where(analysis["KMEANS_K2_ORIGINAL"] == low_original_label, 1, 2)
analysis["PROFILE_CODE"] = np.where(
    analysis["PROFILE_ID"] == 1,
    "CHIEN_LUOC_TUONG_DOI_THAP",
    "CHIEN_LUOC_TUONG_DOI_CAO",
)

profile_mapping = pd.DataFrame([
    {
        "KMEANS_K2_ORIGINAL": low_original_label,
        "PROFILE_ID": 1,
        "PROFILE_CODE": "CHIEN_LUOC_TUONG_DOI_THAP",
        "T2_INTENSITY_Z_MEAN": float(original_means.loc[low_original_label]),
    },
    {
        "KMEANS_K2_ORIGINAL": high_original_label,
        "PROFILE_ID": 2,
        "PROFILE_CODE": "CHIEN_LUOC_TUONG_DOI_CAO",
        "T2_INTENSITY_Z_MEAN": float(original_means.loc[high_original_label]),
    },
])
save_csv(profile_mapping, run_dir / "01_anh_xa_nhan_cum_3A_sang_ho_so_3B.csv")
print(profile_mapping.to_string(index=False))

In [ ]:
balanced_columns = [f"Z_BALANCED_{scale}" for scale in CLUSTER_FEATURES]
X_balanced = analysis[balanced_columns].to_numpy(dtype=float)
labels_original = analysis["KMEANS_K2_ORIGINAL"].to_numpy(dtype=int)
analysis["SILHOUETTE"] = silhouette_samples(X_balanced, labels_original, metric="euclidean")

centers = {
    label: X_balanced[labels_original == label].mean(axis=0)
    for label in sorted(np.unique(labels_original))
}
assigned_distance = []
other_distance = []
for row, label in zip(X_balanced, labels_original):
    other = next(value for value in centers if value != label)
    assigned_distance.append(float(np.linalg.norm(row - centers[label])))
    other_distance.append(float(np.linalg.norm(row - centers[other])))
analysis["DISTANCE_TO_ASSIGNED_CENTROID"] = assigned_distance
analysis["DISTANCE_TO_OTHER_CENTROID"] = other_distance
analysis["CENTROID_DISTANCE_MARGIN"] = analysis["DISTANCE_TO_OTHER_CENTROID"] - analysis["DISTANCE_TO_ASSIGNED_CENTROID"]

analysis["SILHOUETTE_BAND"] = pd.cut(
    analysis["SILHOUETTE"],
    bins=[-np.inf, 0, 0.10, 0.25, 0.50, np.inf],
    labels=["AM", "GAN_RANH_GIOI", "YEU", "VUA", "RO"],
    right=False,
)

recomputed_silhouette = float(analysis["SILHOUETTE"].mean())
reference_row = decision_3a.loc[decision_3a["k"] == SELECTED_K]
if len(reference_row) != 1:
    raise AssertionError("The 3A selection table must have exactly one K=2 row.")
reference_silhouette = float(reference_row.iloc[0]["silhouette"])
if not math.isclose(recomputed_silhouette, reference_silhouette, rel_tol=0, abs_tol=SILHOUETTE_REFERENCE_TOLERANCE):
    raise AssertionError(
        f"Recomputed silhouette {recomputed_silhouette} differs from phase 3A {reference_silhouette}."
    )

silhouette_summary_rows = []
for profile_id in ["TOAN_BO", 1, 2]:
    block = analysis if profile_id == "TOAN_BO" else analysis[analysis["PROFILE_ID"] == profile_id]
    silhouette_summary_rows.append({
        "PROFILE_ID": profile_id,
        "n": int(len(block)),
        "mean": float(block["SILHOUETTE"].mean()),
        "median": float(block["SILHOUETTE"].median()),
        "min": float(block["SILHOUETTE"].min()),
        "q1": float(block["SILHOUETTE"].quantile(0.25)),
        "q3": float(block["SILHOUETTE"].quantile(0.75)),
        "max": float(block["SILHOUETTE"].max()),
        "negative_n": int((block["SILHOUETTE"] < 0).sum()),
        "negative_pct": float((block["SILHOUETTE"] < 0).mean()),
        "below_0_10_n": int((block["SILHOUETTE"] < 0.10).sum()),
        "below_0_10_pct": float((block["SILHOUETTE"] < 0.10).mean()),
    })
silhouette_summary = pd.DataFrame(silhouette_summary_rows)
silhouette_bands = (
    analysis.groupby(["PROFILE_ID", "SILHOUETTE_BAND"], observed=False)
    .size().rename("n").reset_index()
)
silhouette_bands["pct_within_profile"] = silhouette_bands.groupby("PROFILE_ID")["n"].transform(lambda values: values / values.sum())

save_csv(analysis[[
    "RESPONDENT_ID", "KMEANS_K2_ORIGINAL", "PROFILE_ID", "PROFILE_CODE", "T2_INTENSITY_Z",
    "SILHOUETTE", "SILHOUETTE_BAND", "DISTANCE_TO_ASSIGNED_CENTROID",
    "DISTANCE_TO_OTHER_CENTROID", "CENTROID_DISTANCE_MARGIN",
]], run_dir / "02_silhouette_tung_nguoi.csv")
save_csv(silhouette_summary, run_dir / "03_tom_tat_silhouette.csv")
save_csv(silhouette_bands, run_dir / "04_phan_phoi_muc_silhouette.csv")
print(silhouette_summary.to_string(index=False))

In [ ]:
internal_rows = []
for scale in CLUSTER_FEATURES:
    z_column = f"Z_EQUAL_{scale}"
    low_raw = analysis.loc[analysis["PROFILE_ID"] == 1, scale]
    high_raw = analysis.loc[analysis["PROFILE_ID"] == 2, scale]
    low_z = analysis.loc[analysis["PROFILE_ID"] == 1, z_column]
    high_z = analysis.loc[analysis["PROFILE_ID"] == 2, z_column]
    internal_rows.append({
        "scale": scale,
        "tier": "TANG_I" if scale in TIER1_SCALES else "TANG_II",
        "n_low": int(len(low_raw)),
        "mean_raw_low": float(low_raw.mean()),
        "sd_raw_low": float(low_raw.std(ddof=1)),
        "mean_z_low": float(low_z.mean()),
        "n_high": int(len(high_raw)),
        "mean_raw_high": float(high_raw.mean()),
        "sd_raw_high": float(high_raw.std(ddof=1)),
        "mean_z_high": float(high_z.mean()),
        "difference_raw_high_minus_low": float(high_raw.mean() - low_raw.mean()),
        "difference_z_high_minus_low": float(high_z.mean() - low_z.mean()),
        "hedges_g_descriptive": hedges_g(high_raw, low_raw),
    })
internal_profiles = pd.DataFrame(internal_rows)

deltas = internal_profiles["difference_z_high_minus_low"].to_numpy()
shape_summary = pd.DataFrame([{
    "all_differences_same_direction": bool(np.all(deltas >= 0) or np.all(deltas <= 0)),
    "positive_difference_count": int((deltas > 0).sum()),
    "negative_difference_count": int((deltas < 0).sum()),
    "mean_absolute_z_difference": float(np.mean(np.abs(deltas))),
    "sd_z_differences": float(np.std(deltas, ddof=1)),
    "min_z_difference": float(deltas.min()),
    "max_z_difference": float(deltas.max()),
    "interpretation": (
        "CHU_YEU_PHAN_CHIA_MUC_DO" if (np.sum(deltas > 0) >= len(deltas) - 1 or np.sum(deltas < 0) >= len(deltas) - 1)
        else "CO_DAU_HIEU_HINH_DANG_HO_SO_KHAC_NHAU"
    ),
}])

# VIF diagnoses redundancy but is not a K-Means prerequisite.
X_equal = analysis[[f"Z_EQUAL_{scale}" for scale in CLUSTER_FEATURES]].to_numpy(dtype=float)
vif_rows = []
for index, scale in enumerate(CLUSTER_FEATURES):
    y = X_equal[:, index]
    X_other = np.delete(X_equal, index, axis=1)
    r_squared = LinearRegression().fit(X_other, y).score(X_other, y)
    vif_rows.append({"scale": scale, "r_squared_from_other_features": float(r_squared), "VIF": float(1 / (1 - r_squared))})
vif_table = pd.DataFrame(vif_rows).sort_values("VIF", ascending=False)
correlation_11 = analysis[CLUSTER_FEATURES].corr(method="spearman")

save_csv(internal_profiles, run_dir / "05_ho_so_noi_bo_11_bien.csv")
save_csv(shape_summary, run_dir / "06_tom_tat_hinh_dang_ho_so.csv")
save_csv(vif_table, run_dir / "07_VIF_11_bien.csv")
correlation_11.to_csv(run_dir / "08_tuong_quan_spearman_11_bien.csv", encoding="utf-8-sig")
print(internal_profiles.to_string(index=False))
print("\nProfile shape summary:\n", shape_summary.to_string(index=False))
print("\nVIF:\n", vif_table.to_string(index=False))

In [ ]:
bootstrap_rng = np.random.default_rng(RANDOM_SEED + 1000)
external_rows = []
for scale in EXTERNAL_SCALES:
    low = analysis.loc[analysis["PROFILE_ID"] == 1, scale].to_numpy(dtype=float)
    high = analysis.loc[analysis["PROFILE_ID"] == 2, scale].to_numpy(dtype=float)
    welch = ttest_ind(high, low, equal_var=False)
    mann = mannwhitneyu(high, low, alternative="two-sided")
    effect = hedges_g(high, low)
    bootstrap = bootstrap_difference_and_g(high, low, BOOTSTRAP_RESAMPLES, bootstrap_rng)
    external_rows.append({
        "scale": scale,
        "tier": "TANG_III" if scale in ["LPSN", "SFN", "TECN"] else "TANG_IV",
        "n_low": int(len(low)),
        "mean_low": float(low.mean()),
        "sd_low": float(low.std(ddof=1)),
        "n_high": int(len(high)),
        "mean_high": float(high.mean()),
        "sd_high": float(high.std(ddof=1)),
        "difference_high_minus_low": float(high.mean() - low.mean()),
        "hedges_g": effect,
        "effect_magnitude_heuristic": effect_label(effect),
        "welch_t": float(welch.statistic),
        "welch_p": float(welch.pvalue),
        "mann_whitney_u": float(mann.statistic),
        "mann_whitney_p": float(mann.pvalue),
        **bootstrap,
    })
external_tests = pd.DataFrame(external_rows)
external_tests["welch_q_bh"] = benjamini_hochberg(external_tests["welch_p"])
external_tests["mann_whitney_q_bh"] = benjamini_hochberg(external_tests["mann_whitney_p"])
external_tests["ci_g_excludes_zero"] = (
    (external_tests["hedges_g_ci_low"] > 0) & (external_tests["hedges_g_ci_high"] > 0)
) | (
    (external_tests["hedges_g_ci_low"] < 0) & (external_tests["hedges_g_ci_high"] < 0)
)

# Multivariate test uses Euclidean distances across five standardized scales.
external_matrix = analysis[EXTERNAL_SCALES].to_numpy(dtype=float)
external_z = (external_matrix - external_matrix.mean(axis=0)) / external_matrix.std(axis=0, ddof=0)
permutation_rng = np.random.default_rng(RANDOM_SEED + 2000)
pseudo_f, r_squared, permutation_p = permanova_two_groups(
    external_z,
    analysis["PROFILE_ID"].to_numpy(),
    PERMUTATION_RESAMPLES,
    permutation_rng,
)
permanova_table = pd.DataFrame([{
    "variables": ",".join(EXTERNAL_SCALES),
    "n": int(len(analysis)),
    "groups": 2,
    "pseudo_F": pseudo_f,
    "R_squared": r_squared,
    "permutation_p": permutation_p,
    "permutations": PERMUTATION_RESAMPLES,
    "distance": "EUCLIDEAN_ON_Z_SCORES",
}])

save_csv(external_tests, run_dir / "09_kiem_dinh_ngoai_tung_thang.csv")
save_csv(permanova_table, run_dir / "10_PERMANOVA_tong_the_Tang_III_IV.csv")
print(external_tests.to_string(index=False))
print("\nPERMANOVA:\n", permanova_table.to_string(index=False))

In [ ]:
k2_3a = reference_row.iloc[0]
k2_stability = stability.loc[stability["k"] == SELECTED_K]
if len(k2_stability) != 1:
    raise AssertionError("The 3A stability table must have exactly one K=2 row.")
k2_stability = k2_stability.iloc[0]

stable_partition = bool(k2_stability["ARI_mean"] >= 0.80 and k2_stability["Jaccard_min_cluster_mean"] >= 0.75)
weak_separation = bool(recomputed_silhouette < 0.25)
external_supported = bool(
    permutation_p < 0.05
    and ((external_tests["welch_q_bh"] < 0.05) & (external_tests["hedges_g"].abs() >= 0.20)).sum() >= 2
)

if stable_partition and external_supported and weak_separation:
    conditional_conclusion = "GIU_K2_NHU_PHAN_KHUC_THUC_DUNG_NHUNG_KHONG_GOI_LA_HAI_CUM_TU_NHIEN_RO_RANG"
elif stable_partition and external_supported:
    conditional_conclusion = "K2_CO_DO_ON_DINH_VA_GIA_TRI_KIEM_DINH_NGOAI"
elif stable_partition and not external_supported:
    conditional_conclusion = "K2_ON_DINH_NHUNG_CHUA_CO_GIA_TRI_KIEM_DINH_NGOAI"
else:
    conditional_conclusion = "CHUA_DU_BANG_CHUNG_GIU_K2"

evidence_summary = pd.DataFrame([
    {"aspect": "DO_TACH_BIET", "indicator": "Mean silhouette", "value": recomputed_silhouette, "assessment": "YEU" if weak_separation else "TU_VUA_TRO_LEN"},
    {"aspect": "RANH_GIOI", "indicator": "Negative silhouette fraction", "value": float((analysis["SILHOUETTE"] < 0).mean()), "assessment": "MO_TA_KHONG_PHAI_NGUONG_LOAI"},
    {"aspect": "DO_ON_DINH", "indicator": "Mean resampled ARI", "value": float(k2_stability["ARI_mean"]), "assessment": "DAT" if stable_partition else "CHUA_DAT"},
    {"aspect": "DO_ON_DINH", "indicator": "Minimum cluster Jaccard", "value": float(k2_stability["Jaccard_min_cluster_mean"]), "assessment": "DAT" if stable_partition else "CHUA_DAT"},
    {"aspect": "KIEM_DINH_NGOAI", "indicator": "PERMANOVA R-squared", "value": r_squared, "assessment": "DOC_KEM_P_VALUE_VA_HEDGES_G"},
    {"aspect": "KIEM_DINH_NGOAI", "indicator": "PERMANOVA permutation p value", "value": permutation_p, "assessment": "CO_Y_NGHIA" if permutation_p < 0.05 else "KHONG_CO_Y_NGHIA"},
    {"aspect": "KET_LUAN", "indicator": "Conditional conclusion", "value": np.nan, "assessment": conditional_conclusion},
])
save_csv(evidence_summary, run_dir / "11_tom_tat_bang_chung_K2.csv")

print(evidence_summary.to_string(index=False))
print("\nConditional conclusion:", conditional_conclusion)

In [ ]:
# Standard silhouette plot.
fig, axis = plt.subplots(figsize=(8.5, 5.5))
y_lower = 10
for profile_id, shade in [(1, "0.75"), (2, "0.35")]:
    values = np.sort(analysis.loc[analysis["PROFILE_ID"] == profile_id, "SILHOUETTE"].to_numpy())
    y_upper = y_lower + len(values)
    axis.fill_betweenx(np.arange(y_lower, y_upper), 0, values, facecolor=shade, edgecolor="black", linewidth=0.4)
    axis.text(
        0.012,
        y_lower + 0.5 * len(values),
        f"Profile {profile_id}",
        va="center",
        color="black" if profile_id == 1 else "white",
        fontweight="bold",
    )
    y_lower = y_upper + 10
axis.axvline(recomputed_silhouette, color="black", linestyle="--", label=f"Mean = {recomputed_silhouette:.3f}")
axis.axvline(0, color="black", linewidth=0.8)
axis.set_title("Silhouette distribution at K = 2", fontweight="bold")
axis.set_xlabel("Respondent silhouette")
axis.set_ylabel("Respondents per profile")
axis.set_yticks([])
axis.legend(frameon=False)
axis.grid(axis="x", color="0.88")
fig.tight_layout()
fig.savefig(run_dir / "12_bieu_do_silhouette_tung_nguoi.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# Eleven-scale profile plotted in z-score units.
fig, axis = plt.subplots(figsize=(10.5, 5.5))
x = np.arange(len(CLUSTER_FEATURES))
for profile_id, marker, linestyle in [(1, "o", "--"), (2, "s", "-")]:
    block = analysis[analysis["PROFILE_ID"] == profile_id]
    means = np.array([block[f"Z_EQUAL_{scale}"].mean() for scale in CLUSTER_FEATURES])
    ses = np.array([block[f"Z_EQUAL_{scale}"].std(ddof=1) / math.sqrt(len(block)) for scale in CLUSTER_FEATURES])
    axis.plot(x, means, color="black", marker=marker, linestyle=linestyle, label=f"Profile {profile_id}")
    axis.fill_between(x, means - 1.96 * ses, means + 1.96 * ses, color="0.80" if profile_id == 1 else "0.55", alpha=0.25)
axis.axhline(0, color="black", linewidth=0.8)
axis.set_xticks(x, CLUSTER_FEATURES)
axis.set_ylabel("Mean z-score and 95% confidence interval")
axis.set_title("Two-profile shape on 11 clustering variables", fontweight="bold")
axis.legend(frameon=False)
axis.grid(axis="y", color="0.88")
fig.tight_layout()
fig.savefig(run_dir / "13_bieu_do_ho_so_11_bien.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# Means across the five external validation scales.
fig, axis = plt.subplots(figsize=(9, 5.5))
width = 0.34
for offset, profile_id, shade in [(-width / 2, 1, "0.75"), (width / 2, 2, "0.30")]:
    block = analysis[analysis["PROFILE_ID"] == profile_id]
    means = np.array([block[scale].mean() for scale in EXTERNAL_SCALES])
    ses = np.array([block[scale].std(ddof=1) / math.sqrt(len(block)) for scale in EXTERNAL_SCALES])
    axis.bar(np.arange(len(EXTERNAL_SCALES)) + offset, means, width, color=shade, edgecolor="black", label=f"Profile {profile_id}")
    axis.errorbar(np.arange(len(EXTERNAL_SCALES)) + offset, means, yerr=1.96 * ses, fmt="none", color="black", capsize=3)
axis.set_xticks(np.arange(len(EXTERNAL_SCALES)), EXTERNAL_SCALES)
axis.set_ylim(1, 5)
axis.set_ylabel("Mean score on the 1 to 5 scale")
axis.set_title("Tiers III and IV by profile", fontweight="bold")
axis.legend(frameon=False)
axis.grid(axis="y", color="0.88")
fig.tight_layout()
fig.savefig(run_dir / "14_bieu_do_kiem_dinh_ngoai_Tang_III_IV.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# Forest plot Hedges g.
plot_data = external_tests.iloc[::-1].reset_index(drop=True)
fig, axis = plt.subplots(figsize=(8.5, 5.2))
y = np.arange(len(plot_data))
effects = plot_data["hedges_g"].to_numpy()
lower = plot_data["hedges_g_ci_low"].to_numpy()
upper = plot_data["hedges_g_ci_high"].to_numpy()
axis.errorbar(effects, y, xerr=[effects - lower, upper - effects], fmt="o", color="black", capsize=4)
axis.axvline(0, color="black", linewidth=0.8)
axis.set_yticks(y, plot_data["scale"])
axis.set_xlabel("Hedges g: higher-strategy minus lower-strategy profile")
axis.set_title("Effect sizes on external validation scales", fontweight="bold")
axis.grid(axis="x", color="0.88")
fig.tight_layout()
fig.savefig(run_dir / "15_forest_plot_Hedges_g.png", dpi=200, bbox_inches="tight")
plt.close(fig)

print("Four figures were saved.")

In [ ]:
negative_n = int((analysis["SILHOUETTE"] < 0).sum())
negative_pct = float((analysis["SILHOUETTE"] < 0).mean())
boundary_n = int((analysis["SILHOUETTE"] < 0.10).sum())
boundary_pct = float((analysis["SILHOUETTE"] < 0.10).mean())
meaningful_external_n = int(((external_tests["welch_q_bh"] < 0.05) & (external_tests["hedges_g"].abs() >= 0.20)).sum())

report_text = f'''# Phase 3B: evaluation of the two-profile solution

## Objective and integrity

Evaluate K-Means at K = 2 using individual silhouette values, the 11 clustering scales, and five external validation scales. Phase 3A contains {len(scores)} scored respondents; the primary clustering sample contains {len(analysis)} complete ENP respondents. Respondent IDs and SHA-256 manifests agree. Recomputed silhouette is {recomputed_silhouette:.6f}, compared with {reference_silhouette:.6f} in 3A.

## Separation and profile shape

- Mean silhouette: {recomputed_silhouette:.3f}.
- Respondents with negative silhouette: {negative_n} ({negative_pct:.2%}).
- Respondents with silhouette below 0.10: {boundary_n} ({boundary_pct:.2%}).
- Profile 1 has relatively lower tier II strategy use than profile 2 in this sample.
- Shape interpretation: {shape_summary.iloc[0]['interpretation']}.

Silhouette below 0.25 signals weak geometric separation. The 11 clustering scales formed the labels, so their p values are not used as external validation.

## External assessment

- PERMANOVA pseudo-F: {pseudo_f:.4f}; R-squared: {r_squared:.4f}.
- Permutation p value: {permutation_p:.6f} from {PERMUTATION_RESAMPLES} permutations.
- External scales meeting q < 0.05 and absolute Hedges g >= 0.20: {meaningful_external_n}/{len(EXTERNAL_SCALES)}.

Tier III and IV scales were not used to form profiles, but they came from the same self-report survey. They do not constitute an independent validation sample.

## Conditional conclusion

`{conditional_conclusion}`

If K = 2 is retained as a practical segmentation, report its weak geometric separation. Profile labels describe relative levels within this sample, not inherently distinct learner types.

Read the silhouette summary, internal profile table, external scale tests, PERMANOVA, K = 2 evidence table, and figures 12 through 15.
'''

with open(run_dir / "16_huong_dan_doc_ket_qua.md", "w", encoding="utf-8") as handle:
    handle.write(report_text)

run_config = {
    "created_utc": run_stamp,
    "python_version": platform.python_version(),
    "selected_k": SELECTED_K,
    "cluster_column": CLUSTER_COLUMN,
    "expected_primary_n": EXPECTED_PRIMARY_N,
    "random_seed": RANDOM_SEED,
    "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
    "permutation_resamples": PERMUTATION_RESAMPLES,
    "fast_test": FAST_TEST,
    "important_limitation": "Held-out scales were not used for clustering but came from the same self-report survey.",
}
with open(run_dir / "17_cau_hinh_lan_chay.json", "w", encoding="utf-8") as handle:
    json.dump(run_config, handle, ensure_ascii=False, indent=2)

readback_checks = []
for filename, expected_rows in [
    ("01_anh_xa_nhan_cum_3A_sang_ho_so_3B.csv", 2),
    ("02_silhouette_tung_nguoi.csv", EXPECTED_PRIMARY_N),
    ("03_tom_tat_silhouette.csv", 3),
    ("05_ho_so_noi_bo_11_bien.csv", len(CLUSTER_FEATURES)),
    ("09_kiem_dinh_ngoai_tung_thang.csv", len(EXTERNAL_SCALES)),
    ("10_PERMANOVA_tong_the_Tang_III_IV.csv", 1),
    ("11_tom_tat_bang_chung_K2.csv", 7),
]:
    frame = pd.read_csv(run_dir / filename, encoding="utf-8-sig")
    passed = len(frame) == expected_rows
    readback_checks.append({"file": filename, "expected_rows": expected_rows, "actual_rows": len(frame), "passed": passed})
    if not passed:
        raise AssertionError(f"Readback failed: {filename}")
readback = pd.DataFrame(readback_checks)
save_csv(readback, run_dir / "18_kiem_tra_doc_lai_dau_ra.csv")

manifest_rows = []
for path in sorted(run_dir.iterdir()):
    if path.is_file() and path.name != "19_manifest_sha256.csv":
        manifest_rows.append({"file": path.name, "size_bytes": path.stat().st_size, "sha256": sha256_file(path)})
manifest = pd.DataFrame(manifest_rows)
save_csv(manifest, run_dir / "19_manifest_sha256.csv")

archive_path = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir.parent, base_dir=run_dir.name)

print(report_text)
print("Phase 3B complete.")
print("Results directory:", run_dir)
print("ZIP file:", archive_path)

In [ ]:
PHASE3_3B_DIR = str(run_dir)
print('3B results:', PHASE3_3B_DIR)

## 3C. Compare algorithms and sensitivity configurations


In [ ]:
OUTPUT_BASE_DIR = str(output_root / '3C')

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Python dependencies are ready.')
K_VALUES = list(range(2, 5)) if FAST_TEST else list(range(2, 11))
RANDOM_SEED = 20260911
KMEANS_N_INIT = 20 if FAST_TEST else 100
KMEANS_MAX_ITER_OLD = 300
KMEANS_MAX_ITER_AUDIT = 500
KMEANS_TOL = 1e-4
SEED_AUDIT_RUNS = 8 if FAST_TEST else 100
GAP_REFERENCE_SAMPLES = 3 if FAST_TEST else 30
STABILITY_RESAMPLES = 4 if FAST_TEST else 500
STABILITY_SUBSAMPLE_FRACTION = 0.80
PERMANOVA_PERMUTATIONS = 49 if FAST_TEST else 999
MIN_CLUSTER_PROPORTION = 0.05


In [ ]:
import hashlib
import json
import math
import platform
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    pairwise_distances,
    silhouette_samples,
    silhouette_score,
)
from sklearn.metrics.cluster import contingency_matrix
from sklearn.preprocessing import RobustScaler, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.edgecolor': 'black',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'text.color': 'black',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

SCALE_ITEMS = {
    'ENP': [f'ENP{i:02d}' for i in range(1, 7)],
    'PTED': [f'PTED{i:02d}' for i in range(1, 7)],
    'MEU': [f'MEU{i:02d}' for i in range(1, 6)],
    'REH': [f'REH{i:02d}' for i in range(1, 4)],
    'ELA': [f'ELA{i:02d}' for i in range(1, 4)],
    'ORG': [f'ORG{i:02d}' for i in range(1, 4)],
    'CRT': [f'CRT{i:02d}' for i in range(1, 4)],
    'MSR': [f'MSR{i:02d}' for i in range(1, 5)],
    'TSEM': [f'TSEM{i:02d}' for i in range(1, 4)],
    'VLS': [f'VLS{i:02d}' for i in range(1, 7)],
    'DEL': [f'DEL{i:02d}' for i in range(1, 8)],
    'LPSN': [f'LPSN{i:02d}' for i in range(1, 6)],
    'SFN': [f'SFN{i:02d}' for i in range(1, 5)],
    'TECN': [f'TECN{i:02d}' for i in range(1, 7)],
    'PU': [f'PU{i:02d}' for i in range(1, 5)],
    'BI': [f'BI{i:02d}' for i in range(1, 4)],
}
ALL_ITEMS = [item for items in SCALE_ITEMS.values() for item in items]
TIER1_SCALES = ['ENP', 'PTED', 'MEU']
TIER1_NO_ENP = ['PTED', 'MEU']
TIER2_SCALES = ['REH', 'ELA', 'ORG', 'CRT', 'MSR', 'TSEM', 'VLS', 'DEL']
PRIMARY_FEATURES = TIER1_SCALES + TIER2_SCALES
SENSITIVITY_FEATURES = TIER1_NO_ENP + TIER2_SCALES
HELD_OUT_SCALES = ['LPSN', 'SFN', 'TECN', 'PU', 'BI']

# Primary weights give each tier a total contribution of one.
WEIGHTS_TIER_BALANCED = {
    **{name: 1 / len(TIER1_SCALES) for name in TIER1_SCALES},
    **{name: 1 / len(TIER2_SCALES) for name in TIER2_SCALES},
}

# Use estimated PTED:MEU ratios only with matching phase 2 output.
# ENP keeps neutral weight; all three tier I scales are neutral without phase 2 output.
if SEM_BETA is not None:
    other_mean = (SEM_BETA['PTED'] + SEM_BETA['MEU']) / 2
    if other_mean <= 0:
        raise ValueError('The combined PTED and MEU beta must be positive.')
    WEIGHTS_SEM_SENSITIVITY = {
        'ENP': 1 / 3, 'PTED': SEM_BETA['PTED'] / other_mean / 3,
        'MEU': SEM_BETA['MEU'] / other_mean / 3,
        **{name: 1 / len(TIER2_SCALES) for name in TIER2_SCALES},
    }
else:
    WEIGHTS_SEM_SENSITIVITY = None


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_csv_robust(path):
    encodings = ['utf-8-sig', 'utf-8', 'cp1258', 'cp1252', 'latin1']
    last_error = None
    for encoding in encodings:
        try:
            with open(path, 'r', encoding=encoding) as handle:
                first_line = handle.readline()
            separators = sorted([',', ';', '\t'], key=lambda value: first_line.count(value), reverse=True)
            for separator in separators:
                try:
                    frame = pd.read_csv(path, encoding=encoding, sep=separator)
                    if frame.shape[1] > 1:
                        return frame, encoding, separator
                except Exception as error:
                    last_error = error
        except Exception as error:
            last_error = error
    raise RuntimeError(f'Cannot read CSV: {last_error}')


def save_csv(frame, path):
    frame.to_csv(path, index=False, encoding='utf-8-sig')


def cluster_counts(labels):
    _, counts = np.unique(labels, return_counts=True)
    return counts


def wcss_common(X, labels):
    total = 0.0
    for label in np.unique(labels):
        block = X[labels == label]
        center = block.mean(axis=0)
        total += float(np.square(block - center).sum())
    return max(total, np.finfo(float).tiny)


def align_labels(reference, candidate):
    matrix = contingency_matrix(reference, candidate)
    row_ind, col_ind = linear_sum_assignment(-matrix)
    mapping = {candidate_label: reference_label for reference_label, candidate_label in zip(row_ind, col_ind)}
    return np.array([mapping.get(label, label) for label in candidate], dtype=int)


def cluster_jaccard(reference, candidate):
    aligned = align_labels(reference, candidate)
    values = []
    for label in np.unique(reference):
        a = reference == label
        b = aligned == label
        union = np.logical_or(a, b).sum()
        values.append(float(np.logical_and(a, b).sum() / union) if union else np.nan)
    return values


def pam_build_swap(distance_matrix, k, max_iter=100, tol=1e-10):
    '''Deterministic PAM uses BUILD and SWAP with the supplied distance matrix.'''
    D = np.asarray(distance_matrix, dtype=float)
    n = D.shape[0]
    if D.shape != (n, n) or not np.allclose(D, D.T, atol=1e-10):
        raise ValueError('Invalid PAM distance matrix.')
    if not 2 <= k < n:
        raise ValueError('Invalid number of clusters k.')

    medoids = [int(np.argmin(D.sum(axis=1)))]
    nearest = D[:, medoids[0]].copy()
    while len(medoids) < k:
        candidates = [idx for idx in range(n) if idx not in medoids]
        gains = [float(np.maximum(nearest - D[:, idx], 0).sum()) for idx in candidates]
        best = candidates[int(np.argmax(gains))]
        medoids.append(best)
        nearest = np.minimum(nearest, D[:, best])

    medoids = np.array(medoids, dtype=int)
    converged = False
    iterations = 0
    for iterations in range(1, max_iter + 1):
        current_cost = float(D[:, medoids].min(axis=1).sum())
        best_cost = current_cost
        best_swap = None
        non_medoids = np.setdiff1d(np.arange(n), medoids, assume_unique=False)
        for position in range(k):
            retained = np.delete(medoids, position)
            retained_distance = D[:, retained].min(axis=1) if len(retained) else np.full(n, np.inf)
            for candidate in non_medoids:
                candidate_cost = float(np.minimum(retained_distance, D[:, candidate]).sum())
                if candidate_cost < best_cost - tol:
                    best_cost = candidate_cost
                    best_swap = (position, int(candidate))
        if best_swap is None:
            converged = True
            break
        medoids[best_swap[0]] = best_swap[1]

    labels = np.argmin(D[:, medoids], axis=1)
    for cluster_id, medoid_index in enumerate(medoids):
        labels[medoid_index] = cluster_id
    counts = cluster_counts(labels)
    if len(counts) != k or counts.min() == 0:
        raise RuntimeError('PAM produced an empty cluster after assignment.')
    objective = float(D[np.arange(n), medoids[labels]].sum())
    return labels.astype(int), medoids, objective, iterations, converged


def native_silhouette(config, labels):
    if config['algorithm'] == 'KMEANS':
        return float(silhouette_score(config['X'], labels, metric='euclidean'))
    return float(silhouette_score(config['D'], labels, metric='precomputed'))


def fit_configuration(config, k, seed):
    if config['algorithm'] == 'KMEANS':
        model = KMeans(
            n_clusters=k,
            init='k-means++',
            n_init=config['n_init'],
            max_iter=config['max_iter'],
            tol=config['tol'],
            random_state=seed,
            algorithm='lloyd',
        )
        labels = model.fit_predict(config['X'])
        return {
            'labels': labels.astype(int),
            'model': model,
            'objective': float(model.inertia_),
            'n_iter': int(model.n_iter_),
            'converged': bool(model.n_iter_ < config['max_iter']),
            'empty_cluster': bool(len(np.unique(labels)) != k),
        }
    labels, medoids, objective, n_iter, converged = pam_build_swap(
        config['D'], k, max_iter=config['max_iter'], tol=config['tol']
    )
    return {
        'labels': labels,
        'medoids': medoids,
        'objective': objective,
        'n_iter': n_iter,
        'converged': converged,
        'empty_cluster': bool(len(np.unique(labels)) != k),
    }


def predict_from_subset(config, subset, fitted):
    if config['algorithm'] == 'KMEANS':
        return fitted['model'].predict(config['X']).astype(int)
    medoids_global = subset[fitted['medoids']]
    labels = np.argmin(config['D'][:, medoids_global], axis=1).astype(int)
    for cluster_id, medoid_index in enumerate(medoids_global):
        labels[medoid_index] = cluster_id
    return labels


def permanova_pseudo_f(X, labels):
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)
    n = len(labels)
    groups = np.unique(labels)
    grand = X.mean(axis=0)
    ss_total = float(np.square(X - grand).sum())
    ss_within = 0.0
    for group in groups:
        block = X[labels == group]
        ss_within += float(np.square(block - block.mean(axis=0)).sum())
    ss_between = ss_total - ss_within
    df_between = len(groups) - 1
    df_within = n - len(groups)
    pseudo_f = (ss_between / df_between) / (ss_within / df_within)
    r2 = ss_between / ss_total if ss_total > 0 else np.nan
    return float(pseudo_f), float(r2)


def hedges_g(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    n1, n2 = len(x), len(y)
    pooled = math.sqrt(((n1 - 1) * x.var(ddof=1) + (n2 - 1) * y.var(ddof=1)) / (n1 + n2 - 2))
    if pooled == 0:
        return 0.0
    d = (x.mean() - y.mean()) / pooled
    correction = 1 - 3 / (4 * (n1 + n2) - 9)
    return float(correction * d)


def welch_anova(groups):
    '''One-way Welch ANOVA returning F, p, df1, and df2.'''
    arrays = [np.asarray(group, dtype=float) for group in groups]
    sizes = np.array([len(group) for group in arrays], dtype=float)
    means = np.array([group.mean() for group in arrays], dtype=float)
    variances = np.array([group.var(ddof=1) for group in arrays], dtype=float)
    variances = np.maximum(variances, np.finfo(float).eps)
    weights = sizes / variances
    weight_sum = weights.sum()
    weighted_mean = float(np.sum(weights * means) / weight_sum)
    k = len(arrays)
    term = float(np.sum(((1 - weights / weight_sum) ** 2) / (sizes - 1)))
    numerator = float(np.sum(weights * (means - weighted_mean) ** 2) / (k - 1))
    correction = 1 + (2 * (k - 2) / (k**2 - 1)) * term
    statistic = numerator / correction
    df1 = k - 1
    df2 = (k**2 - 1) / (3 * term) if term > 0 else np.inf
    p_value = float(stats.f.sf(statistic, df1, df2))
    return statistic, p_value, float(df1), float(df2)


def fdr_bh(p_values):
    '''Benjamini-Hochberg correction preserving the original order.'''
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted_ranked = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = np.minimum(adjusted_ranked, 1.0)
    return adjusted

In [ ]:
input_path = Path(INPUT_CSV_PATH)
output_base = Path(OUTPUT_BASE_DIR)
if not input_path.is_file():
    raise FileNotFoundError(f'Input file not found: {input_path}')
output_base.mkdir(parents=True, exist_ok=True)

run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = output_base / f'notebook_3C_so_sanh_kmeans_{run_stamp}'
run_dir.mkdir(parents=False, exist_ok=False)

input_hash_before = sha256_file(input_path)
df, detected_encoding, detected_separator = read_csv_robust(input_path)
df.columns = [str(column).replace('\ufeff', '').strip() for column in df.columns]

if df.columns.duplicated().any():
    raise AssertionError('Input column names must be unique.')
missing_columns = [column for column in ALL_ITEMS if column not in df.columns]
if missing_columns:
    raise AssertionError(f'Required columns missing: {missing_columns}')
if 'RESPONDENT_ID' not in df.columns:
    df.insert(0, 'RESPONDENT_ID', [f'ROW_{row:08d}' for row in range(1, len(df) + 1)])
if df['RESPONDENT_ID'].isna().any() or df['RESPONDENT_ID'].duplicated().any():
    raise AssertionError('RESPONDENT_ID must be present and unique.')

invalid_rows = []
for column in ALL_ITEMS:
    converted = pd.to_numeric(df[column], errors='coerce')
    nonempty_original = df[column].notna() & df[column].astype(str).str.strip().ne('')
    failed_conversion = nonempty_original & converted.isna()
    invalid_value = converted.notna() & ~converted.isin([1, 2, 3, 4, 5])
    if failed_conversion.any():
        invalid_rows.append({'column': column, 'problem': 'cannot be parsed as a number', 'count': int(failed_conversion.sum())})
    if invalid_value.any():
        invalid_rows.append({'column': column, 'problem': 'outside the 1 to 5 range', 'count': int(invalid_value.sum())})
    df[column] = converted
if invalid_rows:
    raise AssertionError(f'Invalid values: {invalid_rows}')

enp_items = SCALE_ITEMS['ENP']
enp_all_missing = df[enp_items].isna().all(axis=1)
enp_partial_missing = df[enp_items].isna().any(axis=1) & ~enp_all_missing
non_enp_items = [item for item in ALL_ITEMS if item not in enp_items]
if enp_partial_missing.any():
    raise AssertionError('An ENP response block is only partly observed.')
if int(df[non_enp_items].isna().sum().sum()) != 0:
    raise AssertionError('An item outside ENP is missing.')

complete_71 = int((~df[ALL_ITEMS].isna().any(axis=1)).sum())
if sha256_file(input_path) != input_hash_before:
    raise AssertionError('Input changed during reading.')

input_audit = {
    'input_path': str(input_path),
    'sha256': input_hash_before,
    'encoding': detected_encoding,
    'separator': {',': 'comma', ';': 'semicolon', '\t': 'tab'}.get(detected_separator, detected_separator),
    'rows': int(len(df)),
    'columns': int(df.shape[1]),
    'likert_items': int(len(ALL_ITEMS)),
    'complete_71': complete_71,
    'full_enp_missing': int(enp_all_missing.sum()),
    'missing_outside_enp': int(df[non_enp_items].isna().sum().sum()),
    'pu_bi_missing': int(df[SCALE_ITEMS['PU'] + SCALE_ITEMS['BI']].isna().sum().sum()),
}
with open(run_dir / '00_kiem_tra_dau_vao.json', 'w', encoding='utf-8') as handle:
    json.dump(input_audit, handle, ensure_ascii=False, indent=2)
print(json.dumps(input_audit, ensure_ascii=False, indent=2))
print('Results directory:', run_dir)
EXPECTED_ROWS = len(df)
EXPECTED_COMPLETE_ENP = int((~df[ALL_ITEMS].isna().any(axis=1)).sum())
EXPECTED_ENP_MISSING = int(enp_all_missing.sum())
if EXPECTED_COMPLETE_ENP < max(K_VALUES) + 1 or EXPECTED_ROWS < max(K_VALUES) + 1:
    raise ValueError("Too few complete ENP profiles for the configured K range.")


In [ ]:
score_df = pd.DataFrame({'RESPONDENT_ID': df['RESPONDENT_ID'].astype(str)})
for scale, items in SCALE_ITEMS.items():
    score_df[scale] = df[items].mean(axis=1, skipna=False)
score_df['PTED_EASE'] = 6 - score_df['PTED']
for optional in ['GENDER', 'FIELD_OF_STUDY', 'S0_ENGLISH_NECESSARY']:
    if optional in df.columns:
        score_df[optional] = df[optional]

primary_mask = score_df[PRIMARY_FEATURES].notna().all(axis=1)
primary_scores = score_df.loc[primary_mask, ['RESPONDENT_ID', *PRIMARY_FEATURES, *HELD_OUT_SCALES]].reset_index(drop=True)
sensitivity_scores = score_df[['RESPONDENT_ID', *SENSITIVITY_FEATURES]].copy().reset_index(drop=True)
if len(primary_scores) != EXPECTED_COMPLETE_ENP:
    raise AssertionError('Primary sample size differs from ENP-complete profiles.')
if len(sensitivity_scores) != EXPECTED_ROWS:
    raise AssertionError('Sensitivity sample size differs from input rows.')

standard_scaler = StandardScaler()
Z = standard_scaler.fit_transform(primary_scores[PRIMARY_FEATURES])
robust_scaler = RobustScaler(quantile_range=(25, 75))
R = robust_scaler.fit_transform(primary_scores[PRIMARY_FEATURES])

tier_weights = np.array([WEIGHTS_TIER_BALANCED[name] for name in PRIMARY_FEATURES], dtype=float)
sem_weights = np.array([WEIGHTS_SEM_SENSITIVITY[name] for name in PRIMARY_FEATURES], dtype=float) if SEM_BETA else None

X_M0 = Z * np.sqrt(tier_weights)
X_M2 = Z * np.sqrt(sem_weights) if SEM_BETA else None
X_M3 = Z * tier_weights
X_M4 = R * np.sqrt(tier_weights)
X_M5 = Z * sem_weights if SEM_BETA else None

model_configs = {
    'M0_KMEANS_CU': {
        'algorithm': 'KMEANS', 'X': X_M0, 'native_metric': 'euclidean_squared',
        'n_init': KMEANS_N_INIT, 'max_iter': KMEANS_MAX_ITER_OLD, 'tol': KMEANS_TOL,
        'scientific_role': 'Historical reference configuration',
    },
    'M1_KMEANS_KIEM_TOAN': {
        'algorithm': 'KMEANS', 'X': X_M0, 'native_metric': 'euclidean_squared',
        'n_init': KMEANS_N_INIT, 'max_iter': KMEANS_MAX_ITER_AUDIT, 'tol': KMEANS_TOL,
        'scientific_role': 'Fully specified and audited primary configuration',
    },
    'M2_KMEANS_TRONG_SO_TANG_I': {
        'algorithm': 'KMEANS', 'X': X_M2, 'native_metric': 'weighted_euclidean_squared',
        'n_init': KMEANS_N_INIT, 'max_iter': KMEANS_MAX_ITER_AUDIT, 'tol': KMEANS_TOL,
        'scientific_role': 'SEM-weighted sensitivity' if SEM_BETA else 'Neutral-weight comparator',
    },
    'M3_PAM_MANHATTAN_CAN_BANG': {
        'algorithm': 'PAM', 'X': X_M3, 'D': pairwise_distances(X_M3, metric='manhattan'),
        'native_metric': 'weighted_manhattan', 'n_init': 1,
        'max_iter': 100, 'tol': 1e-10,
        'scientific_role': 'L1 distance sensitivity with reduced outlier influence',
    },
    'M4_KMEANS_ROBUST': {
        'algorithm': 'KMEANS', 'X': X_M4, 'native_metric': 'robust_scaled_euclidean_squared',
        'n_init': KMEANS_N_INIT, 'max_iter': KMEANS_MAX_ITER_AUDIT, 'tol': KMEANS_TOL,
        'scientific_role': 'Robust scaling sensitivity',
    },
    'M5_PAM_MANHATTAN_TANG_I': {
        'algorithm': 'PAM', 'X': X_M5, 'D': pairwise_distances(X_M5 if SEM_BETA else X_M3, metric='manhattan'),
        'native_metric': 'sem_weighted_manhattan', 'n_init': 1,
        'max_iter': 100, 'tol': 1e-10,
        'scientific_role': 'L1 and SEM sensitivity' if SEM_BETA else 'L1 neutral-weight comparator',
    },
}

if SEM_BETA is None:
    model_configs.pop('M2_KMEANS_TRONG_SO_TANG_I')
    model_configs.pop('M5_PAM_MANHATTAN_TANG_I')
    print('Phase 2 results unavailable: omit M2/M5 without inventing SEM weights.')

weight_rows = []
for feature in PRIMARY_FEATURES:
    weight_rows.append({
        'feature': feature,
        'tier': 'Tier I' if feature in TIER1_SCALES else 'Tier II',
        'tier_balanced_weight': WEIGHTS_TIER_BALANCED[feature],
        'sem_sensitivity_weight': WEIGHTS_SEM_SENSITIVITY[feature] if SEM_BETA else np.nan,
        'beta_source': SEM_BETA.get(feature, np.nan) if SEM_BETA else np.nan,
        'note': 'Neutral ENP' if feature == 'ENP' else (('SEM-aligned' if SEM_BETA else 'Neutral when phase 2 is unavailable') if feature in ['PTED', 'MEU'] else 'Balanced tier II'),
    })
weight_table = pd.DataFrame(weight_rows)

space_export = primary_scores[['RESPONDENT_ID', *PRIMARY_FEATURES]].copy()
for index, feature in enumerate(PRIMARY_FEATURES):
    space_export[f'Z_{feature}'] = Z[:, index]
    space_export[f'M0_{feature}'] = X_M0[:, index]
    if SEM_BETA:
        space_export[f'M2_{feature}'] = X_M2[:, index]
    space_export[f'M3_{feature}'] = X_M3[:, index]
    space_export[f'M4_{feature}'] = X_M4[:, index]
    if SEM_BETA:
        space_export[f'M5_{feature}'] = X_M5[:, index]

sens_scaler = StandardScaler()
Z_sens = sens_scaler.fit_transform(sensitivity_scores[SENSITIVITY_FEATURES])
sens_weights = np.array([
    1 / len(TIER1_NO_ENP) if feature in TIER1_NO_ENP else 1 / len(TIER2_SCALES)
    for feature in SENSITIVITY_FEATURES
])
X_sensitivity = Z_sens * np.sqrt(sens_weights)

save_csv(score_df, run_dir / '01_diem_16_thang.csv')
save_csv(weight_table, run_dir / '02_bang_trong_so_va_vai_tro.csv')
save_csv(space_export, run_dir / '03_khong_gian_dau_vao_MAU_CHINH.csv')
save_csv(sensitivity_scores, run_dir / '04_dau_vao_do_nhay_MAU_DAY_DU_khong_ENP.csv')

print('Primary sample:', primary_scores.shape)
print('Sensitivity sample:', sensitivity_scores.shape)
print(weight_table.to_string(index=False))

In [ ]:
scan_rows = []
fit_store = {}
for model_index, (model_name, config) in enumerate(model_configs.items()):
    fit_store[model_name] = {}
    for k in K_VALUES:
        fitted = fit_configuration(config, k, RANDOM_SEED + model_index * 1000 + k)
        labels = fitted['labels']
        counts = cluster_counts(labels)
        fit_store[model_name][k] = fitted
        scan_rows.append({
            'model': model_name,
            'algorithm': config['algorithm'],
            'scientific_role': config['scientific_role'],
            'native_metric': config['native_metric'],
            'k': k,
            'silhouette_native': native_silhouette(config, labels),
            'silhouette_common': float(silhouette_score(X_M0, labels, metric='euclidean')),
            'calinski_harabasz_common': float(calinski_harabasz_score(X_M0, labels)),
            'davies_bouldin_common': float(davies_bouldin_score(X_M0, labels)),
            'wcss_common': wcss_common(X_M0, labels),
            'native_objective': fitted['objective'],
            'n_iter': fitted['n_iter'],
            'converged': fitted['converged'],
            'empty_cluster': fitted['empty_cluster'],
            'min_cluster_n': int(counts.min()),
            'max_cluster_n': int(counts.max()),
            'min_cluster_pct': float(counts.min() / len(labels)),
        })

scan_df = pd.DataFrame(scan_rows)
if scan_df['empty_cluster'].any():
    raise AssertionError('A configuration produced an empty cluster.')
if not scan_df['converged'].all():
    print('WARNING: an optimizer reached its iteration limit; inspect the metrics.')

# Rank internal metrics within each model; exclude tiers III and IV from K selection.
candidate_rows = []
for model_name, block in scan_df.groupby('model', sort=False):
    block = block.copy()
    eligible = block['min_cluster_pct'] >= MIN_CLUSTER_PROPORTION
    working = block.loc[eligible].copy() if eligible.any() else block.copy()
    working['rank_sil_native'] = working['silhouette_native'].rank(ascending=False, method='min')
    working['rank_sil_common'] = working['silhouette_common'].rank(ascending=False, method='min')
    working['rank_ch_common'] = working['calinski_harabasz_common'].rank(ascending=False, method='min')
    working['rank_db_common'] = working['davies_bouldin_common'].rank(ascending=True, method='min')
    working['mean_internal_rank'] = working[
        ['rank_sil_native', 'rank_sil_common', 'rank_ch_common', 'rank_db_common']
    ].mean(axis=1)
    selected = working.sort_values(['mean_internal_rank', 'k']).iloc[0]
    candidate_rows.append({
        'model': model_name,
        'candidate_k': int(selected['k']),
        'mean_internal_rank': float(selected['mean_internal_rank']),
        'selection_rule': 'Mean rank across four metrics; reject undersized clusters when an eligible solution exists',
        'automatic_final_decision': 'KHONG_TU_DONG_KET_LUAN_TOI_UU',
    })
candidate_df = pd.DataFrame(candidate_rows)

save_csv(scan_df, run_dir / '05_chi_so_quet_K_M0_den_M5.csv')
save_csv(candidate_df, run_dir / '06_ung_vien_K_theo_tung_cau_hinh.csv')
print(candidate_df.to_string(index=False))

In [ ]:
rng_gap = np.random.default_rng(RANDOM_SEED + 10000)
gap_rows = []
for model_name, config in model_configs.items():
    if config['algorithm'] != 'KMEANS':
        continue
    mins = config['X'].min(axis=0)
    maxs = config['X'].max(axis=0)
    for k in K_VALUES:
        observed = fit_store[model_name][k]['objective']
        reference_logs = []
        for iteration in range(GAP_REFERENCE_SAMPLES):
            reference = rng_gap.uniform(mins, maxs, size=config['X'].shape)
            ref_model = KMeans(
                n_clusters=k, init='k-means++', n_init=min(20, KMEANS_N_INIT),
                max_iter=config['max_iter'], tol=config['tol'],
                random_state=RANDOM_SEED + 20000 + iteration + k * 100,
                algorithm='lloyd',
            ).fit(reference)
            reference_logs.append(math.log(max(float(ref_model.inertia_), np.finfo(float).tiny)))
        reference_logs = np.asarray(reference_logs)
        gap_rows.append({
            'model': model_name,
            'k': k,
            'gap': float(reference_logs.mean() - math.log(max(observed, np.finfo(float).tiny))),
            'gap_se': float(reference_logs.std(ddof=1) * math.sqrt(1 + 1 / GAP_REFERENCE_SAMPLES)) if len(reference_logs) > 1 else np.nan,
        })
gap_df = pd.DataFrame(gap_rows)
gap_df['gap_rule_candidate'] = False
for model_name, indices in gap_df.groupby('model').groups.items():
    ordered = gap_df.loc[indices].sort_values('k')
    for left_pos in range(len(ordered) - 1):
        left = ordered.iloc[left_pos]
        right = ordered.iloc[left_pos + 1]
        if left['gap'] >= right['gap'] - right['gap_se']:
            gap_df.loc[left.name, 'gap_rule_candidate'] = True
            break

sensitivity_rows = []
sensitivity_fits = {}
for k in K_VALUES:
    model = KMeans(
        n_clusters=k, init='k-means++', n_init=KMEANS_N_INIT,
        max_iter=KMEANS_MAX_ITER_AUDIT, tol=KMEANS_TOL,
        random_state=RANDOM_SEED + 30000 + k, algorithm='lloyd',
    )
    labels = model.fit_predict(X_sensitivity)
    sensitivity_fits[k] = {'model': model, 'labels': labels}
    counts = cluster_counts(labels)
    sensitivity_rows.append({
        'k': k,
        'silhouette': float(silhouette_score(X_sensitivity, labels)),
        'calinski_harabasz': float(calinski_harabasz_score(X_sensitivity, labels)),
        'davies_bouldin': float(davies_bouldin_score(X_sensitivity, labels)),
        'wcss': float(model.inertia_),
        'min_cluster_n': int(counts.min()),
        'min_cluster_pct': float(counts.min() / len(labels)),
    })
sensitivity_df = pd.DataFrame(sensitivity_rows)

primary_index_in_sensitivity = score_df.reset_index().set_index('RESPONDENT_ID').loc[primary_scores['RESPONDENT_ID'], 'index'].to_numpy()
sensitivity_agreement_rows = []
for k in K_VALUES:
    labels_n_primary = fit_store['M0_KMEANS_CU'][k]['labels']
    labels_n_all_on_primary = sensitivity_fits[k]['labels'][primary_index_in_sensitivity]
    sensitivity_agreement_rows.append({
        'k': k,
        'ARI_M0_MAU_CHINH_with_ENP_vs_MAU_DAY_DU_without_ENP': float(adjusted_rand_score(labels_n_primary, labels_n_all_on_primary)),
    })
sensitivity_agreement_df = pd.DataFrame(sensitivity_agreement_rows)

save_csv(gap_df, run_dir / '07_gap_statistic_cac_cau_hinh_kmeans.csv')
save_csv(sensitivity_df, run_dir / '08_do_nhay_MAU_DAY_DU_khong_ENP.csv')
save_csv(sensitivity_agreement_df, run_dir / '09_ARI_MAU_CHINH_so_voi_MAU_DAY_DU.csv')
print('Gap statistic and complete-sample sensitivity finished.')

In [ ]:
seed_rows = []
for model_index, (model_name, config) in enumerate(model_configs.items()):
    if config['algorithm'] != 'KMEANS':
        continue
    candidate_k = int(candidate_df.loc[candidate_df['model'] == model_name, 'candidate_k'].iloc[0])
    reference_labels = fit_store[model_name][candidate_k]['labels']
    for run in range(SEED_AUDIT_RUNS):
        seed = RANDOM_SEED + 40000 + model_index * 1000 + run
        model = KMeans(
            n_clusters=candidate_k, init='k-means++', n_init=1,
            max_iter=config['max_iter'], tol=config['tol'],
            random_state=seed, algorithm='lloyd',
        ).fit(config['X'])
        seed_rows.append({
            'model': model_name,
            'candidate_k': candidate_k,
            'run': run + 1,
            'seed': seed,
            'inertia': float(model.inertia_),
            'n_iter': int(model.n_iter_),
            'converged': bool(model.n_iter_ < config['max_iter']),
            'ARI_vs_production': float(adjusted_rand_score(reference_labels, model.labels_)),
        })
seed_detail = pd.DataFrame(seed_rows)
seed_summary = seed_detail.groupby(['model', 'candidate_k'], as_index=False).agg(
    inertia_best=('inertia', 'min'),
    inertia_median=('inertia', 'median'),
    inertia_worst=('inertia', 'max'),
    ARI_mean=('ARI_vs_production', 'mean'),
    ARI_min=('ARI_vs_production', 'min'),
    iterations_max=('n_iter', 'max'),
    convergence_rate=('converged', 'mean'),
)
save_csv(seed_detail, run_dir / '10_kiem_toan_seed_chi_tiet.csv')
save_csv(seed_summary, run_dir / '11_kiem_toan_seed_tom_tat.csv')
print(seed_summary.to_string(index=False))

In [ ]:
stability_rng = np.random.default_rng(RANDOM_SEED + 50000)
stability_rows = []
n_primary = len(primary_scores)
subsample_n = int(round(n_primary * STABILITY_SUBSAMPLE_FRACTION))

for model_index, (model_name, config) in enumerate(model_configs.items()):
    candidate_k = int(candidate_df.loc[candidate_df['model'] == model_name, 'candidate_k'].iloc[0])
    reference_labels = fit_store[model_name][candidate_k]['labels']
    for iteration in range(STABILITY_RESAMPLES):
        subset = np.sort(stability_rng.choice(n_primary, size=subsample_n, replace=False))
        if config['algorithm'] == 'KMEANS':
            subset_config = dict(config)
            subset_config['X'] = config['X'][subset]
            subset_config['n_init'] = min(20, KMEANS_N_INIT)
        else:
            subset_config = dict(config)
            subset_config['X'] = config['X'][subset]
            subset_config['D'] = config['D'][np.ix_(subset, subset)]
        fitted = fit_configuration(
            subset_config, candidate_k,
            RANDOM_SEED + 60000 + model_index * 10000 + iteration,
        )
        candidate_labels = predict_from_subset(config, subset, fitted)
        jaccards = cluster_jaccard(reference_labels, candidate_labels)
        row = {
            'model': model_name,
            'candidate_k': candidate_k,
            'iteration': iteration + 1,
            'ARI': float(adjusted_rand_score(reference_labels, candidate_labels)),
        }
        for cluster_index, value in enumerate(jaccards, 1):
            row[f'JACCARD_CLUSTER_{cluster_index}'] = value
        stability_rows.append(row)

stability_detail = pd.DataFrame(stability_rows)
stability_summary_rows = []
for (model_name, candidate_k), block in stability_detail.groupby(['model', 'candidate_k'], sort=False):
    jaccard_columns = [column for column in block.columns if column.startswith('JACCARD_CLUSTER_') and block[column].notna().any()]
    cluster_means = block[jaccard_columns].mean()
    stability_summary_rows.append({
        'model': model_name,
        'candidate_k': int(candidate_k),
        'ARI_mean': float(block['ARI'].mean()),
        'ARI_median': float(block['ARI'].median()),
        'ARI_p025': float(block['ARI'].quantile(0.025)),
        'ARI_p975': float(block['ARI'].quantile(0.975)),
        'Jaccard_mean_all_clusters': float(cluster_means.mean()),
        'Jaccard_min_cluster_mean': float(cluster_means.min()),
    })
stability_summary = pd.DataFrame(stability_summary_rows)
save_csv(stability_detail, run_dir / '12_on_dinh_lay_mau_lai_chi_tiet.csv')
save_csv(stability_summary, run_dir / '13_on_dinh_lay_mau_lai_tom_tat.csv')
print(stability_summary.to_string(index=False))

In [ ]:
comparison_rows = []
m0_candidate_k = int(candidate_df.loc[candidate_df['model'] == 'M0_KMEANS_CU', 'candidate_k'].iloc[0])
m0_candidate_labels = fit_store['M0_KMEANS_CU'][m0_candidate_k]['labels']

labels_export = pd.DataFrame({'RESPONDENT_ID': primary_scores['RESPONDENT_ID']})
profile_rows = []
silhouette_person_rows = []

for model_name, config in model_configs.items():
    candidate_k = int(candidate_df.loc[candidate_df['model'] == model_name, 'candidate_k'].iloc[0])
    labels = fit_store[model_name][candidate_k]['labels']
    labels_export[f'{model_name}_K{candidate_k}'] = labels + 1
    same_k_m0 = fit_store['M0_KMEANS_CU'][candidate_k]['labels']
    scan_row = scan_df[(scan_df['model'] == model_name) & (scan_df['k'] == candidate_k)].iloc[0]
    stability_row = stability_summary[stability_summary['model'] == model_name].iloc[0]
    comparison_rows.append({
        'model': model_name,
        'scientific_role': config['scientific_role'],
        'candidate_k': candidate_k,
        'silhouette_native': scan_row['silhouette_native'],
        'silhouette_common': scan_row['silhouette_common'],
        'calinski_harabasz_common': scan_row['calinski_harabasz_common'],
        'davies_bouldin_common': scan_row['davies_bouldin_common'],
        'min_cluster_n': scan_row['min_cluster_n'],
        'min_cluster_pct': scan_row['min_cluster_pct'],
        'stability_ARI_mean': stability_row['ARI_mean'],
        'stability_Jaccard_min': stability_row['Jaccard_min_cluster_mean'],
        'ARI_vs_M0_at_same_K': float(adjusted_rand_score(same_k_m0, labels)),
        'ARI_vs_M0_primary_candidate': float(adjusted_rand_score(m0_candidate_labels, labels)),
        'automatic_final_decision': 'KHONG_TU_DONG_KET_LUAN_TOI_UU',
    })
    native_sample_values = (
        silhouette_samples(config['X'], labels, metric='euclidean')
        if config['algorithm'] == 'KMEANS'
        else silhouette_samples(config['D'], labels, metric='precomputed')
    )
    for index, respondent_id in enumerate(primary_scores['RESPONDENT_ID']):
        silhouette_person_rows.append({
            'RESPONDENT_ID': respondent_id,
            'model': model_name,
            'candidate_k': candidate_k,
            'cluster': int(labels[index] + 1),
            'silhouette_native': float(native_sample_values[index]),
        })
    for cluster in range(candidate_k):
        members = labels == cluster
        for feature_index, feature in enumerate(PRIMARY_FEATURES):
            raw_value = float(primary_scores.loc[members, feature].mean())
            profile_rows.append({
                'model': model_name,
                'candidate_k': candidate_k,
                'cluster': cluster + 1,
                'n': int(members.sum()),
                'feature': feature,
                'mean_raw': raw_value,
                'mean_z': float(Z[members, feature_index].mean()),
                'mean_display': float(6 - raw_value) if feature == 'PTED' else raw_value,
                'display_direction': 'Higher means less difficulty' if feature == 'PTED' else 'Higher means a greater level',
            })

comparison_df = pd.DataFrame(comparison_rows)
profile_df = pd.DataFrame(profile_rows)
silhouette_person_df = pd.DataFrame(silhouette_person_rows)

save_csv(comparison_df, run_dir / '14_bang_so_sanh_cau_hinh.csv')
save_csv(labels_export, run_dir / '15_nhan_cum_ung_vien_MAU_CHINH.csv')
save_csv(profile_df, run_dir / '16_ho_so_trung_tam_cum.csv')
save_csv(silhouette_person_df, run_dir / '17_silhouette_tung_nguoi.csv')
print(comparison_df.to_string(index=False))

In [ ]:
heldout_raw = primary_scores[HELD_OUT_SCALES].to_numpy(float)
heldout_z = StandardScaler().fit_transform(heldout_raw)
permanova_rng = np.random.default_rng(RANDOM_SEED + 70000)
permanova_rows = []
univariate_rows = []

for model_name in model_configs:
    candidate_k = int(candidate_df.loc[candidate_df['model'] == model_name, 'candidate_k'].iloc[0])
    labels = fit_store[model_name][candidate_k]['labels']
    observed_f, r2 = permanova_pseudo_f(heldout_z, labels)
    exceed = 0
    for _ in range(PERMANOVA_PERMUTATIONS):
        permuted = permanova_rng.permutation(labels)
        permuted_f, _ = permanova_pseudo_f(heldout_z, permuted)
        exceed += int(permuted_f >= observed_f)
    p_value = (exceed + 1) / (PERMANOVA_PERMUTATIONS + 1)
    permanova_rows.append({
        'model': model_name,
        'candidate_k': candidate_k,
        'pseudo_F': observed_f,
        'R2': r2,
        'permutations': PERMANOVA_PERMUTATIONS,
        'p_value': p_value,
        'interpretation_limit': 'Observed association difference, not causation',
    })

    for scale in HELD_OUT_SCALES:
        values = primary_scores[scale].to_numpy(float)
        groups = [values[labels == cluster] for cluster in np.unique(labels)]
        if candidate_k == 2:
            welch = stats.ttest_ind(groups[0], groups[1], equal_var=False)
            nonparam = stats.mannwhitneyu(groups[0], groups[1], alternative='two-sided')
            effect = hedges_g(groups[0], groups[1])
            effect_name = 'Hedges_g_cluster1_minus_cluster2'
            param_stat, param_p = float(welch.statistic), float(welch.pvalue)
        else:
            welch_statistic, welch_p, _, _ = welch_anova(groups)
            nonparam = stats.kruskal(*groups)
            h_value = float(nonparam.statistic)
            effect = max(0.0, (h_value - candidate_k + 1) / (len(values) - candidate_k))
            effect_name = 'epsilon_squared_Kruskal'
            param_stat, param_p = float(welch_statistic), float(welch_p)
        univariate_rows.append({
            'model': model_name,
            'candidate_k': candidate_k,
            'scale': scale,
            'parametric_test': 'Welch_t' if candidate_k == 2 else 'Welch_ANOVA',
            'parametric_statistic': param_stat,
            'parametric_p': param_p,
            'nonparametric_test': 'Mann_Whitney' if candidate_k == 2 else 'Kruskal_Wallis',
            'nonparametric_statistic': float(nonparam.statistic),
            'nonparametric_p': float(nonparam.pvalue),
            'effect_name': effect_name,
            'effect_value': effect,
        })

permanova_df = pd.DataFrame(permanova_rows)
univariate_df = pd.DataFrame(univariate_rows)
univariate_df['parametric_p_FDR'] = np.nan
univariate_df['nonparametric_p_FDR'] = np.nan
for model_name, indices in univariate_df.groupby('model').groups.items():
    univariate_df.loc[indices, 'parametric_p_FDR'] = fdr_bh(
        univariate_df.loc[indices, 'parametric_p']
    )
    univariate_df.loc[indices, 'nonparametric_p_FDR'] = fdr_bh(
        univariate_df.loc[indices, 'nonparametric_p']
    )

save_csv(permanova_df, run_dir / '18_PERMANOVA_TierIII_IV.csv')
save_csv(univariate_df, run_dir / '19_kiem_dinh_tung_thang_TierIII_IV.csv')
print(permanova_df.to_string(index=False))

In [ ]:
# 20. Metrics by K and configuration.
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
metric_specs = [
    ('silhouette_native', 'Silhouette in model distance space', 'Higher is better'),
    ('silhouette_common', 'Silhouette in the shared M0 space', 'Higher is better'),
    ('calinski_harabasz_common', 'Calinski-Harabasz in shared space', 'Higher is better'),
    ('davies_bouldin_common', 'Davies-Bouldin in shared space', 'Lower is better'),
]
for axis, (column, title, ylabel) in zip(axes.ravel(), metric_specs):
    for model_name, block in scan_df.groupby('model', sort=False):
        axis.plot(block['k'], block[column], marker='o', linewidth=1.2, label=model_name)
    axis.set_title(title, fontweight='bold')
    axis.set_xlabel('Number of clusters K')
    axis.set_ylabel(ylabel)
    axis.grid(color='0.88')
axes[0, 0].legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(run_dir / '20_chi_so_theo_K_va_cau_hinh.png', dpi=220, bbox_inches='tight')
plt.close(fig)

# 21. Heatmap Silhouette chung.
heat = scan_df.pivot(index='model', columns='k', values='silhouette_common')
fig, axis = plt.subplots(figsize=(10, 5.5))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='Greys', cbar_kws={'label': 'Silhouette chung'}, ax=axis)
axis.set_title('Silhouette comparison in the shared M0 space', fontweight='bold')
axis.set_xlabel('Number of clusters K')
axis.set_ylabel('Configuration')
fig.tight_layout()
fig.savefig(run_dir / '21_heatmap_silhouette_chung.png', dpi=220, bbox_inches='tight')
plt.close(fig)

# 22. Candidate stability.
plot_stability = stability_summary.merge(candidate_df[['model', 'candidate_k']], on=['model', 'candidate_k'])
fig, axis = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(plot_stability))
width = 0.36
axis.bar(x - width / 2, plot_stability['ARI_mean'], width, label='Mean ARI', color='0.25')
axis.bar(x + width / 2, plot_stability['Jaccard_min_cluster_mean'], width, label='Minimum cluster Jaccard', color='0.65')
axis.set_xticks(x, plot_stability['model'], rotation=30, ha='right')
axis.set_ylim(0, 1.02)
axis.set_ylabel('Stability')
axis.set_title('Candidate K stability by configuration', fontweight='bold')
axis.legend(frameon=False)
axis.grid(axis='y', color='0.88')
fig.tight_layout()
fig.savefig(run_dir / '22_do_on_dinh_cac_cau_hinh.png', dpi=220, bbox_inches='tight')
plt.close(fig)

# 23. PCA projection for six candidates.
pca_projection = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X_M0)
fig, axes = plt.subplots(2, 3, figsize=(14, 8.5))
for axis, model_name in zip(axes.ravel(), model_configs):
    candidate_k = int(candidate_df.loc[candidate_df['model'] == model_name, 'candidate_k'].iloc[0])
    labels = fit_store[model_name][candidate_k]['labels']
    axis.scatter(pca_projection[:, 0], pca_projection[:, 1], c=labels, cmap='tab10', s=16, alpha=0.8, edgecolors='none')
    axis.set_title(f'{model_name}\nK = {candidate_k}', fontweight='bold', fontsize=9)
    axis.set_xlabel('PCA 1')
    axis.set_ylabel('PCA 2')
fig.suptitle('Shared PCA projection of candidate assignments', fontweight='bold')
fig.tight_layout()
fig.savefig(run_dir / '23_PCA_so_sanh_nhan_ung_vien.png', dpi=220, bbox_inches='tight')
plt.close(fig)

# 24. Mean z-score profile map.
models = list(model_configs)
fig, axes = plt.subplots(3, 2, figsize=(13, 13))
for axis, model_name in zip(axes.ravel(), models):
    block = profile_df[profile_df['model'] == model_name].pivot(index='cluster', columns='feature', values='mean_z')[PRIMARY_FEATURES]
    sns.heatmap(block, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1.5, vmax=1.5, cbar=False, ax=axis)
    candidate_k = int(candidate_df.loc[candidate_df['model'] == model_name, 'candidate_k'].iloc[0])
    axis.set_title(f'{model_name}, K = {candidate_k}', fontweight='bold', fontsize=9)
    axis.set_xlabel('Scale')
    axis.set_ylabel('Cluster')
fig.suptitle('Standardized profiles across candidate configurations', fontweight='bold')
fig.tight_layout()
fig.savefig(run_dir / '24_ban_do_ho_so_cum.png', dpi=220, bbox_inches='tight')
plt.close(fig)

# 25. Configurations at candidate K.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
bars = [
    ('silhouette_common', 'Silhouette chung', 'Higher is better'),
    ('stability_ARI_mean', 'Resampled ARI', 'Higher is better'),
    ('ARI_vs_M0_at_same_K', 'ARI relative to M0 at the same K', 'Change relative to the reference pipeline'),
]
for axis, (column, title, ylabel) in zip(axes, bars):
    axis.bar(comparison_df['model'], comparison_df[column], color='0.35')
    axis.set_title(title, fontweight='bold')
    axis.set_ylabel(ylabel)
    axis.tick_params(axis='x', rotation=70)
    axis.grid(axis='y', color='0.88')
fig.tight_layout()
fig.savefig(run_dir / '25_bieu_do_so_sanh_cau_hinh.png', dpi=220, bbox_inches='tight')
plt.close(fig)

# 26. Cluster size by candidate.
size_rows = []
for model_name in model_configs:
    candidate_k = int(candidate_df.loc[candidate_df['model'] == model_name, 'candidate_k'].iloc[0])
    labels = fit_store[model_name][candidate_k]['labels']
    for cluster, count in zip(*np.unique(labels + 1, return_counts=True)):
        size_rows.append({'model': model_name, 'cluster': f'Cluster {cluster}', 'n': int(count)})
size_df = pd.DataFrame(size_rows)
fig, axis = plt.subplots(figsize=(11, 5.5))
sns.barplot(data=size_df, x='model', y='n', hue='cluster', palette='Greys', ax=axis)
axis.set_title('Cluster sizes by candidate configuration', fontweight='bold')
axis.set_xlabel('Configuration')
axis.set_ylabel('Respondents')
axis.tick_params(axis='x', rotation=30)
axis.legend(frameon=False)
axis.grid(axis='y', color='0.88')
fig.tight_layout()
fig.savefig(run_dir / '26_kich_thuoc_cum_ung_vien.png', dpi=220, bbox_inches='tight')
plt.close(fig)

save_csv(size_df, run_dir / '27_kich_thuoc_cum_ung_vien.csv')
print('Seven figures and the cluster size table were saved.')

In [ ]:
best_common_row = comparison_df.sort_values(['silhouette_common', 'stability_ARI_mean'], ascending=[False, False]).iloc[0]
most_stable_row = comparison_df.sort_values(['stability_ARI_mean', 'stability_Jaccard_min'], ascending=[False, False]).iloc[0]
candidate_lines = [
    '| Configuration | Candidate K | Mean rank |',
    '|---|---:|---:|',
]
for _, row in candidate_df.iterrows():
    candidate_lines.append(f"| {row['model']} | {int(row['candidate_k'])} | {row['mean_internal_rank']:.3f} |")
candidate_markdown = '\n'.join(candidate_lines)

guide_text = f'''# Phase 3C: clustering configurations and sensitivity

## Scope

- Input: {len(df)} respondents and {df.shape[1]} columns.
- Primary sample: {len(primary_scores)} respondents with complete ENP responses.
- Sensitivity sample: {len(sensitivity_scores)} respondents without ENP features.
- Clustering scales: {', '.join(PRIMARY_FEATURES)}.
- Tiers III and IV are reserved for external association checks.

M0 reproduces the reference workflow. M1 fully specifies convergence and seeds. M2 and M5 require matching phase 2 estimates; their source status is {SEM_PROVENANCE['type']}. M3 and M5 use PAM Manhattan. M4 evaluates RobustScaler.

## Candidate K values

{candidate_markdown}

Candidate K is based on four internal metrics and the minimum cluster-size criterion; it is not a final scientific decision.

## Configuration comparison

- Highest shared-space silhouette: {best_common_row['model']} at K = {int(best_common_row['candidate_k'])}, silhouette = {best_common_row['silhouette_common']:.3f}.
- Highest mean resampled ARI: {most_stable_row['model']}, ARI = {most_stable_row['stability_ARI_mean']:.3f}.

Review separation, stability, cluster size, center shapes, tier III and IV associations, and ARI relative to M0 before choosing a presentation. Small metric improvements alone do not establish one universally optimal configuration. If silhouette remains around 0.2 to 0.3, report fuzzy cluster boundaries. External associations and PERMANOVA are not causal evidence.
'''
with open(run_dir / '28_huong_dan_doc_ket_qua.md', 'w', encoding='utf-8') as handle:
    handle.write(guide_text)

run_config = {
    'created_utc': run_stamp,
    'python_version': platform.python_version(),
    'random_seed': RANDOM_SEED,
    'k_values': K_VALUES,
    'kmeans_n_init': KMEANS_N_INIT,
    'seed_audit_runs': SEED_AUDIT_RUNS,
    'gap_reference_samples': GAP_REFERENCE_SAMPLES,
    'stability_resamples': STABILITY_RESAMPLES,
    'stability_subsample_fraction': STABILITY_SUBSAMPLE_FRACTION,
    'permanova_permutations': PERMANOVA_PERMUTATIONS,
    'min_cluster_proportion': MIN_CLUSTER_PROPORTION,
    'fast_test': FAST_TEST,
    'weights_tier_balanced': WEIGHTS_TIER_BALANCED,
    'weights_sem_sensitivity': WEIGHTS_SEM_SENSITIVITY,
    'sem_provenance': SEM_PROVENANCE,
    'automatic_final_decision': False,
}
with open(run_dir / '29_cau_hinh_lan_chay.json', 'w', encoding='utf-8') as handle:
    json.dump(run_config, handle, ensure_ascii=False, indent=2)

readback_specs = [
    ('01_diem_16_thang.csv', EXPECTED_ROWS),
    ('03_khong_gian_dau_vao_MAU_CHINH.csv', EXPECTED_COMPLETE_ENP),
    ('05_chi_so_quet_K_M0_den_M5.csv', len(model_configs) * len(K_VALUES)),
    ('06_ung_vien_K_theo_tung_cau_hinh.csv', len(model_configs)),
    ('14_bang_so_sanh_cau_hinh.csv', len(model_configs)),
    ('15_nhan_cum_ung_vien_MAU_CHINH.csv', EXPECTED_COMPLETE_ENP),
    ('18_PERMANOVA_TierIII_IV.csv', len(model_configs)),
]
readback_rows = []
for filename, expected_rows in readback_specs:
    frame = pd.read_csv(run_dir / filename, encoding='utf-8-sig')
    passed = len(frame) == expected_rows
    readback_rows.append({'file': filename, 'expected_rows': expected_rows, 'actual_rows': len(frame), 'passed': passed})
    if not passed:
        raise AssertionError(f'Readback failed: {filename}')
readback_df = pd.DataFrame(readback_rows)
save_csv(readback_df, run_dir / '30_kiem_tra_doc_lai_dau_ra.csv')

if sha256_file(input_path) != input_hash_before:
    raise AssertionError('Input changed before the notebook finished.')

manifest_rows = []
for path in sorted(run_dir.iterdir()):
    if path.is_file() and path.name != '31_manifest_sha256.csv':
        manifest_rows.append({'file': path.name, 'size_bytes': path.stat().st_size, 'sha256': sha256_file(path)})
manifest_df = pd.DataFrame(manifest_rows)
save_csv(manifest_df, run_dir / '31_manifest_sha256.csv')

print(guide_text)
print('Phase 3C complete. No configuration is declared universally optimal.')
print('Results directory:', run_dir)

## Package the phase 3 results


In [ ]:
import zipfile
phase3_dirs = [Path(PHASE3_3A_DIR), Path(PHASE3_3B_DIR), Path(run_dir)]
summary = {
    'total_rows': int(len(df)),
    'primary_rows': int(len(primary_scores)),
    'sensitivity_rows': int(len(sensitivity_scores)),
    'kmeans_k2_silhouette_3a': float(decision_3a.loc[decision_3a['k'] == 2, 'silhouette'].iloc[0]),
    'sem_provenance': SEM_PROVENANCE,
    'fast_test': FAST_TEST,
    'section_dirs': [str(path) for path in phase3_dirs],
}
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
package = output_root / f'phase_03_{stamp}.zip'
with zipfile.ZipFile(package, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for directory in phase3_dirs:
        for file in sorted(directory.rglob('*')):
            if file.is_file():
                archive.write(file, arcname=f'{directory.parent.name}/{directory.name}/{file.relative_to(directory)}')
    archive.writestr('00_tom_tat_lan_chay.json', json.dumps(summary, ensure_ascii=False, indent=2))
with zipfile.ZipFile(package) as check:
    if check.testzip() is not None:
        raise AssertionError('The phase 3 ZIP failed its integrity check.')
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Phase 3 ZIP:', package)
